# Talkie Experiments - Full RunPod Pipeline

This notebook runs all experiments in sequence on RunPod (A100 80GB).

**Estimated time: ~7 hours total**
- Session 1 (Mining + Probing): ~4 hours
- Session 2 (ICL 10 seeds + Qualitative): ~3 hours

Run all cells top to bottom. Each section saves its results independently.

## 0. Setup - Write Updated Files

This section writes all the updated/new files that differ from what's currently on RunPod.

In [1]:
import os
import json
from pathlib import Path

# Set working directory to project root
PROJECT_ROOT = Path(os.getcwd())
print(f"Project root: {PROJECT_ROOT}")
print(f"Files in root: {[f.name for f in PROJECT_ROOT.iterdir() if f.is_file()]}")

Project root: /workspace/Talkie
Files in root: ['run_all_experiments.ipynb', 'qualitative_generations.py', 'probe_analysis.py', 'ocr_ablation.py', 'model_loader.py', 'inspect_checkpoint.py', 'experiment3_probing.py', 'experiment2_icl.py', 'experiment1_syntactic.py', 'diagnose.py', 'debug_nan.py', 'debug_icl2.py', 'debug_icl.py', 'config.py', 'analysis.py']


In [2]:
!pip install tiktoken -q


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
!pip install tiktoken huggingface_hub datasets scikit-learn numpy pandas matplotlib seaborn scipy tqdm -q


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [6]:
config_content = '''"""Central configuration for all experiments."""

from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────
ROOT = Path(__file__).resolve().parent
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
CACHE_DIR = ROOT / "model_cache"

for d in [RESULTS_DIR, FIGURES_DIR, DATA_DIR, CACHE_DIR]:
    d.mkdir(exist_ok=True)

# ── Model IDs ──────────────────────────────────────────────────────────
VINTAGE_MODEL_ID = "talkie-lm/talkie-1930-13b-base"
MODERN_MODEL_ID = "talkie-lm/talkie-web-13b-base"

MODEL_NAMES = {
    VINTAGE_MODEL_ID: "Talkie-1930",
    MODERN_MODEL_ID: "Talkie-Web",
}

# ── Architecture constants ─────────────────────────────────────────────
N_LAYERS = 40
HIDDEN_DIM = 5120
N_HEADS = 40
HEAD_DIM = 128
VOCAB_SIZE = 65536

# ── Experiment 1: BLiMP ────────────────────────────────────────────────
BLIMP_BATCH_SIZE = 16

BLIMP_PHENOMENA = [
    "anaphor_agreement",
    "argument_structure",
    "binding",
    "control_raising",
    "determiner_noun_agreement",
    "ellipsis",
    "filler_gap",
    "irregular_forms",
    "island_effects",
    "npi_licensing",
    "quantifiers",
    "subject_verb_agreement",
]

# ── Experiment 2: ICL ──────────────────────────────────────────────────
ICL_K_VALUES = [0, 1, 2, 4, 8, 16, 32]
ICL_SEEDS = [42, 123, 456, 789, 101, 202, 303, 404, 505, 606]
ICL_MAX_EVAL_SAMPLES = 500

ICL_TASKS = {
    "sst2": {
        "dataset": "stanfordnlp/sst2",
        "split": "validation",
        "input_key": "sentence",
        "label_key": "label",
        "label_names": ["negative", "positive"],
        "description": "Sentiment analysis (movie reviews)",
    },
    "mnli": {
        "dataset": "nyu-mll/multi_nli",
        "split": "validation_matched",
        "input_keys": ["premise", "hypothesis"],
        "label_key": "label",
        "label_names": ["entailment", "neutral", "contradiction"],
        "description": "Natural language inference",
    },
    "tweet_sentiment": {
        "dataset": "cardiffnlp/tweet_eval",
        "config": "sentiment",
        "split": "test",
        "input_key": "text",
        "label_key": "label",
        "label_names": ["negative", "neutral", "positive"],
        "description": "Tweet sentiment classification",
    },
    "tweet_emotion": {
        "dataset": "cardiffnlp/tweet_eval",
        "config": "emotion",
        "split": "test",
        "input_key": "text",
        "label_key": "label",
        "label_names": ["anger", "joy", "optimism", "sadness"],
        "description": "Tweet emotion classification",
    },
}

# ── Experiment 3: Probing ──────────────────────────────────────────────
PROBE_LAYERS = list(range(0, N_LAYERS + 1, 4))
PROBE_CV_FOLDS = 5
PROBE_MAX_SEQ_LEN = 128

# ── OCR ablation ───────────────────────────────────────────────────────
OCR_ERROR_RATES = [0.01, 0.02, 0.05, 0.10]

# ── Device ─────────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DTYPE = torch.float32  # fp32 required - bf16 breaks logits
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    DTYPE = torch.float16
else:
    DEVICE = torch.device("cpu")
    DTYPE = torch.float32
'''

with open('/workspace/Talkie/config.py', 'w') as f:
    f.write(config_content)
print("Written: config.py")

Written: config.py


In [7]:
"""
Download and load Talkie custom models.

Architecture (from reference src/talkie/model.py):
  - Pre-norm GPT with *parameter-free* F.rms_norm (no learnable γ/β)
  - Per-layer scalar gains (attn_gain, mlp_gain, embed_skip)
  - QK-norm + per-head gain on queries
  - RoPE (base=1e6) positional encoding
  - SwiGLU FFN
  - LM head with weight gain
  - Embedding skip-connection (added after MLP, not with attention)

Checkpoint keys follow `_orig_mod.blocks.{i}.*` naming (torch.compile wrapper).
"""

import base64
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import snapshot_download

import config


# ═══════════════════════════════════════════════════════════════════════
#  Architecture  (matches checkpoint key layout exactly)
# ═══════════════════════════════════════════════════════════════════════

@dataclass
class TalkieConfig:
    n_layer: int = 40
    n_head: int = 40
    n_embd: int = 5120
    head_dim: int = 128
    vocab_size: int = 65536
    max_seq_len: int = 2048
    intermediate_size: int = 13696


class HeadGain(nn.Module):
    def __init__(self, n_head: int):
        super().__init__()
        self.head_g = nn.Parameter(torch.ones(n_head))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.head_g.type_as(x).view(1, 1, -1, 1)


class WeightGain(nn.Module):
    def __init__(self):
        super().__init__()
        self.w_g = nn.Parameter(torch.ones(1))

    def forward(self, w: torch.Tensor) -> torch.Tensor:
        return w * self.w_g.type_as(w)


class ActGain(nn.Module):
    def __init__(self, init_value: float):
        super().__init__()
        self.a_g = nn.Parameter(torch.ones(1) * init_value)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.a_g.type_as(x)


# ── RoPE (base=1e6, no learnable params) ─────────────────────────────

def _precompute_rotary(seq_len: int, head_dim: int,
                       base: int = 1_000_000) -> tuple[torch.Tensor, torch.Tensor]:
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
    t = torch.arange(seq_len, dtype=torch.float32)
    freqs = torch.outer(t, inv_freq)
    cos = freqs.cos()[None, :, None, :]
    sin = freqs.sin()[None, :, None, :]
    return cos, sin


def apply_rotary_emb(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    d = x.shape[3] // 2
    x1, x2 = x[..., :d], x[..., d:]
    y1 = x1 * cos + x2 * sin
    y2 = x1 * (-sin) + x2 * cos
    return torch.cat([y1, y2], 3).type_as(x)


# ── Attention (with QK-norm and per-head gain on Q) ──────────────────

class Attention(nn.Module):
    def __init__(self, cfg: TalkieConfig):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.head_dim
        n_state = cfg.n_embd

        self.attn_query = nn.Linear(n_state, n_state, bias=False)
        self.attn_key   = nn.Linear(n_state, n_state, bias=False)
        self.attn_value = nn.Linear(n_state, n_state, bias=False)
        self.attn_resid = nn.Linear(n_state, n_state, bias=False)
        self.head_gain  = HeadGain(cfg.n_head)

    def forward(self, x: torch.Tensor, cos_sin: tuple) -> torch.Tensor:
        B, T, _ = x.shape
        q = self.attn_query(x).view(B, T, self.n_head, self.head_dim)
        k = self.attn_key(x).view(B, T, self.n_head, self.head_dim)
        v = self.attn_value(x).view(B, T, self.n_head, self.head_dim)

        cos, sin = cos_sin
        q = apply_rotary_emb(q, cos, sin)
        k = apply_rotary_emb(k, cos, sin)

        q = F.rms_norm(q, (q.size(-1),))
        k = F.rms_norm(k, (k.size(-1),))
        q = self.head_gain(q)

        y = F.scaled_dot_product_attention(
            q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), is_causal=True
        )
        y = y.transpose(1, 2).contiguous().view(B, T, -1)
        return self.attn_resid(y)


# ── SwiGLU FFN ────────────────────────────────────────────────────────

class MLP(nn.Module):
    def __init__(self, cfg: TalkieConfig):
        super().__init__()
        self.mlp_gate   = nn.Linear(cfg.n_embd, cfg.intermediate_size, bias=False)
        self.mlp_linear = nn.Linear(cfg.n_embd, cfg.intermediate_size, bias=False)
        self.mlp_resid  = nn.Linear(cfg.intermediate_size, cfg.n_embd, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mlp_resid(F.silu(self.mlp_gate(x)) * self.mlp_linear(x))


# ── Transformer block (pre-norm with parameter-free RMS norm) ────────

class TransformerBlock(nn.Module):
    def __init__(self, cfg: TalkieConfig):
        super().__init__()
        self.attn       = Attention(cfg)
        self.mlp        = MLP(cfg)
        self.attn_gain  = ActGain((2 * cfg.n_layer) ** -0.5)
        self.mlp_gain   = ActGain((2 * cfg.n_layer) ** -0.5)
        self.embed_skip = ActGain(0.0)

    def forward(self, e_x: torch.Tensor, x: torch.Tensor,
                cos_sin: tuple) -> torch.Tensor:
        x = x + self.attn_gain(self.attn(F.rms_norm(x, (x.shape[-1],)), cos_sin))
        x = x + self.mlp_gain(self.mlp(F.rms_norm(x, (x.shape[-1],))))
        x = x + self.embed_skip(e_x)
        return x


# ── Full model ────────────────────────────────────────────────────────

class TalkieModel(nn.Module):
    def __init__(self, cfg: TalkieConfig):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layer)])
        self.lm_head = nn.Parameter(torch.zeros(cfg.vocab_size, cfg.n_embd))
        self.lm_head_gain = WeightGain()

        cos, sin = _precompute_rotary(cfg.max_seq_len, cfg.head_dim)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    def forward(self, input_ids: torch.Tensor, output_hidden_states: bool = False,
                last_positions: Optional[torch.Tensor] = None):
        B, T = input_ids.shape
        cos_sin = self.cos[:, :T].to(input_ids.device), self.sin[:, :T].to(input_ids.device)

        x = self.embed(input_ids)
        x = F.rms_norm(x, (x.shape[-1],))
        e_x = x

        hidden_states = [x] if output_hidden_states else None

        for block in self.blocks:
            x = block(e_x, x, cos_sin)
            if output_hidden_states:
                hidden_states.append(x)

        x = F.rms_norm(x, (x.shape[-1],))

        if last_positions is not None:
            x_sel = x[torch.arange(B, device=x.device), last_positions]
            logits = F.linear(x_sel, self.lm_head_gain(self.lm_head)).float()
        else:
            logits = F.linear(x, self.lm_head_gain(self.lm_head)).float()

        if output_hidden_states:
            hidden_states.append(x)
            return logits, tuple(hidden_states)
        return (logits,)


# ═══════════════════════════════════════════════════════════════════════
#  Checkpoint loading
# ═══════════════════════════════════════════════════════════════════════

def _strip_prefix(state_dict: dict, prefix: str = "_orig_mod.") -> dict:
    """Remove torch.compile wrapper prefix from all keys."""
    return {
        (k[len(prefix):] if k.startswith(prefix) else k): v
        for k, v in state_dict.items()
    }


def _detect_config(state_dict: dict) -> TalkieConfig:
    cfg = TalkieConfig()
    for k, v in state_dict.items():
        if not hasattr(v, "shape"):
            continue
        if k == "embed.weight":
            cfg.vocab_size, cfg.n_embd = v.shape
        elif "mlp_gate.weight" in k:
            cfg.intermediate_size = v.shape[0]
        elif "head_gain.head_g" in k:
            cfg.n_head = v.shape[0]

    layer_ids = set()
    for k in state_dict:
        if k.startswith("blocks."):
            idx = k.split(".")[1]
            if idx.isdigit():
                layer_ids.add(int(idx))
    if layer_ids:
        cfg.n_layer = max(layer_ids) + 1
    cfg.head_dim = cfg.n_embd // cfg.n_head

    print(f"[detect] n_layer={cfg.n_layer}  n_embd={cfg.n_embd}  n_head={cfg.n_head}  "
          f"head_dim={cfg.head_dim}  intermediate={cfg.intermediate_size}  vocab={cfg.vocab_size}")
    return cfg


def inspect_checkpoint(ckpt_path):
    raw = torch.load(str(ckpt_path), map_location="cpu", weights_only=False)
    if isinstance(raw, dict):
        for w in ["model", "state_dict"]:
            if w in raw and isinstance(raw[w], dict):
                raw = raw[w]
                print(f"[inspect] Unwrapped '{w}'")
                break
    print(f"\n{'Key':<70} {'Shape':<25} Dtype")
    print("-" * 110)
    for k, v in sorted(raw.items()):
        s = tuple(v.shape) if hasattr(v, "shape") else "scalar"
        d = v.dtype if hasattr(v, "dtype") else type(v).__name__
        print(f"{k:<70} {str(s):<25} {d}")
    print(f"\nTotal keys: {len(raw)}")


# ═══════════════════════════════════════════════════════════════════════
#  Tokeniser
# ═══════════════════════════════════════════════════════════════════════

def load_tiktoken_tokeniser(repo_dir: Path, expected_vocab: int = 65536) -> tiktoken.Encoding:
    """Load tiktoken tokeniser from vocab.txt (base64+rank format)."""
    vocab_path = repo_dir / "vocab.txt"
    if not vocab_path.exists():
        print("[tokeniser] WARNING: No vocab.txt, using cl100k_base fallback.")
        return tiktoken.get_encoding("cl100k_base")

    # Parse base64+rank format: each line is "BASE64_TOKEN RANK"
    mergeable_ranks = {}
    with open(vocab_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) >= 2:
                try:
                    token_bytes = base64.b64decode(parts[0])
                    rank = int(parts[1])
                    if rank < expected_vocab:
                        mergeable_ranks[token_bytes] = rank
                except Exception:
                    continue

    print(f"[tokeniser] Loaded {len(mergeable_ranks)} tokens "
          f"(max rank {max(mergeable_ranks.values()) if mergeable_ranks else 0})")
    return _build_encoding(repo_dir.name, mergeable_ranks)


def _build_encoding(name: str, mergeable_ranks: dict) -> tiktoken.Encoding:
    return tiktoken.Encoding(
        name=f"talkie-{name}",
        pat_str=r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+""",
        mergeable_ranks=mergeable_ranks,
        special_tokens={"<|endoftext|>": len(mergeable_ranks)},
    )


# ═══════════════════════════════════════════════════════════════════════
#  Download
# ═══════════════════════════════════════════════════════════════════════

def download_model_repo(model_id: str) -> Path:
    local_dir = config.CACHE_DIR / model_id.replace("/", "--")
    if local_dir.exists() and any(local_dir.iterdir()):
        print(f"[model_loader] Using cached repo: {local_dir}")
        return local_dir
    print(f"[model_loader] Downloading {model_id} ...")
    snapshot_download(repo_id=model_id, local_dir=str(local_dir))
    return local_dir


def _find_checkpoint(repo_dir: Path) -> Path:
    for pattern in ["*.ckpt", "*.pt", "*.pth", "*.bin"]:
        matches = sorted(repo_dir.glob(pattern), key=lambda p: p.stat().st_size, reverse=True)
        if matches:
            return matches[0]
    raise FileNotFoundError(f"No checkpoint found in {repo_dir}")


# ═══════════════════════════════════════════════════════════════════════
#  ModelWrapper
# ═══════════════════════════════════════════════════════════════════════

class ModelWrapper:
    def __init__(self, model: TalkieModel, tokeniser: tiktoken.Encoding, model_id: str):
        self.model = model
        self.tokeniser = tokeniser
        self.model_id = model_id
        self.name = config.MODEL_NAMES.get(model_id, model_id)
        self.device = config.DEVICE
        self.dtype = config.DTYPE

    @torch.no_grad()
    def log_likelihood(self, text: str) -> float:
        token_ids = self.tokeniser.encode(text)
        if not token_ids:
            return 0.0
        input_ids = torch.tensor([token_ids], device=self.device)
        logits = self.model(input_ids)[0]
        shift_logits = logits[:, :-1, :]
        shift_labels = input_ids[:, 1:]
        log_probs = F.log_softmax(shift_logits, dim=-1)
        return log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1).sum().item()

    @torch.no_grad()
    def log_likelihood_per_token(self, text: str) -> tuple[float, int]:
        token_ids = self.tokeniser.encode(text)
        n = len(token_ids)
        if n <= 1:
            return 0.0, max(n, 1)
        input_ids = torch.tensor([token_ids], device=self.device)
        logits = self.model(input_ids)[0]
        shift_logits = logits[:, :-1, :]
        shift_labels = input_ids[:, 1:]
        log_probs = F.log_softmax(shift_logits, dim=-1)
        return log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1).sum().item(), n

    def score_pair(self, good: str, bad: str) -> dict:
        ll_good, n_good = self.log_likelihood_per_token(good)
        ll_bad, n_bad = self.log_likelihood_per_token(bad)
        return {
            "ll_good": ll_good, "ll_bad": ll_bad,
            "n_tokens_good": n_good, "n_tokens_bad": n_bad,
            "ll_good_norm": ll_good / n_good, "ll_bad_norm": ll_bad / n_bad,
            "correct": ll_good > ll_bad,
            "correct_norm": (ll_good / n_good) > (ll_bad / n_bad),
        }

    @torch.no_grad()
    def batch_log_likelihood(self, texts: list[str],
                             batch_size: int = 256) -> list[tuple[float, int]]:
        all_token_ids = [self.tokeniser.encode(t) for t in texts]
        results = [None] * len(texts)

        for start in range(0, len(texts), batch_size):
            batch_ids = all_token_ids[start:start + batch_size]
            lengths = [len(ids) for ids in batch_ids]
            max_len = max(lengths)

            padded = torch.zeros(len(batch_ids), max_len, dtype=torch.long,
                                 device=self.device)
            for j, ids in enumerate(batch_ids):
                padded[j, :len(ids)] = torch.tensor(ids, dtype=torch.long)

            logits = self.model(padded)[0]
            log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
            targets = padded[:, 1:].unsqueeze(-1)
            token_lps = log_probs.gather(2, targets).squeeze(-1)

            for j, n in enumerate(lengths):
                if n <= 1:
                    results[start + j] = (0.0, max(n, 1))
                else:
                    results[start + j] = (token_lps[j, :n - 1].sum().item(), n)

        return results

    @torch.no_grad()
    def batch_conditional_ll(self, prompts: list[str], completions: list[str],
                             batch_size: int = 32) -> list[tuple[float, int]]:
        """Log P(completion | prompt) for each (prompt, completion) pair.

        Returns list of (conditional_ll, n_completion_tokens).
        """
        prompt_ids = [self.tokeniser.encode(p) for p in prompts]
        full_ids = [self.tokeniser.encode(p + c) for p, c in zip(prompts, completions)]
        results = [None] * len(prompts)

        for start in range(0, len(prompts), batch_size):
            end = min(start + batch_size, len(prompts))
            batch_full = full_ids[start:end]
            batch_prompt_lens = [len(prompt_ids[start + j]) for j in range(end - start)]
            lengths = [len(ids) for ids in batch_full]
            max_len = max(lengths)
            B = len(batch_full)

            padded = torch.zeros(B, max_len, dtype=torch.long, device=self.device)
            for j, ids in enumerate(batch_full):
                padded[j, :len(ids)] = torch.tensor(ids, dtype=torch.long)

            logits = self.model(padded)[0]
            log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
            targets = padded[:, 1:].unsqueeze(-1)
            token_lps = log_probs.gather(2, targets).squeeze(-1)

            for j in range(B):
                prompt_len = batch_prompt_lens[j]
                seq_len = lengths[j]
                n_completion = seq_len - prompt_len
                if n_completion <= 0:
                    results[start + j] = (0.0, 0)
                else:
                    cond_ll = token_lps[j, prompt_len - 1:seq_len - 1].sum().item()
                    results[start + j] = (cond_ll, n_completion)

        return results

    @torch.no_grad()
    def generate(self, prompt: str, max_new_tokens: int = 64,
                 temperature: float = 0.0, stop_tokens: Optional[list[str]] = None) -> str:
        token_ids = self.tokeniser.encode(prompt)
        generated = list(token_ids)
        eos = self.tokeniser.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})
        eos_id = eos[0] if eos else None

        for _ in range(max_new_tokens):
            input_ids = torch.tensor([generated], device=self.device)
            logits = self.model(input_ids)[0]
            next_logits = logits[:, -1, :]
            if temperature <= 0:
                tok = next_logits.argmax(-1).item()
            else:
                tok = torch.multinomial(F.softmax(next_logits / temperature, -1), 1).item()
            if tok == eos_id:
                break
            generated.append(tok)
            if stop_tokens:
                decoded = self.tokeniser.decode(generated[len(token_ids):])
                if any(s in decoded for s in stop_tokens):
                    break
        return self.tokeniser.decode(generated[len(token_ids):])

    @torch.no_grad()
    def extract_hidden_states(self, text: str,
                              layer_indices: Optional[list[int]] = None) -> dict[int, torch.Tensor]:
        if layer_indices is None:
            layer_indices = config.PROBE_LAYERS
        token_ids = self.tokeniser.encode(text)[:config.PROBE_MAX_SEQ_LEN]
        input_ids = torch.tensor([token_ids], device=self.device)
        logits, hidden_states = self.model(input_ids, output_hidden_states=True)
        result = {}
        for idx in layer_indices:
            if idx < len(hidden_states):
                result[idx] = hidden_states[idx].squeeze(0).mean(dim=0).cpu().float()
        return result

    @torch.no_grad()
    def get_next_token_probs(self, prompt: str, target_tokens: list[str]) -> dict[str, float]:
        token_ids = self.tokeniser.encode(prompt)
        input_ids = torch.tensor([token_ids], device=self.device)
        logits = self.model(input_ids)[0]
        probs = F.softmax(logits[:, -1, :], dim=-1).squeeze(0)
        result = {}
        for tok_str in target_tokens:
            ids = self.tokeniser.encode(tok_str)
            result[tok_str] = probs[ids[0]].item() if ids else 0.0
        return result

    @torch.no_grad()
    def batch_next_token_probs(self, prompts: list[str],
                               target_token_ids: list[int],
                               batch_size: int = 32) -> torch.Tensor:
        """Batched next-token probabilities. Returns [n_prompts, n_targets]."""
        all_ids = [self.tokeniser.encode(p) for p in prompts]
        target_idx = torch.tensor(target_token_ids, device=self.device)
        out = torch.zeros(len(prompts), len(target_token_ids))

        for start in range(0, len(prompts), batch_size):
            batch_ids = all_ids[start:start + batch_size]
            lengths = [len(ids) for ids in batch_ids]
            max_len = max(lengths)
            B = len(batch_ids)

            padded = torch.zeros(B, max_len, dtype=torch.long, device=self.device)
            for j, ids in enumerate(batch_ids):
                padded[j, :len(ids)] = torch.tensor(ids, dtype=torch.long)

            last_pos = torch.tensor([l - 1 for l in lengths], device=self.device)
            logits = self.model(padded, last_positions=last_pos)[0]
            probs = F.softmax(logits, dim=-1)
            out[start:start + B] = probs[:, target_idx].cpu()

        return out


# ═══════════════════════════════════════════════════════════════════════
#  Main entry point
# ═══════════════════════════════════════════════════════════════════════

def load_model(model_id: str) -> ModelWrapper:
    repo_dir = download_model_repo(model_id)
    ckpt_path = _find_checkpoint(repo_dir)
    print(f"[model_loader] Checkpoint: {ckpt_path.name} ({ckpt_path.stat().st_size / 1e9:.1f} GB)")

    # Load & unwrap checkpoint
    print("[model_loader] Loading checkpoint into RAM ...")
    raw = torch.load(str(ckpt_path), map_location="cpu", weights_only=False)
    state_dict = raw
    for wrapper in ["model", "state_dict", "model_state_dict"]:
        if isinstance(state_dict, dict) and wrapper in state_dict and isinstance(state_dict[wrapper], dict):
            state_dict = state_dict[wrapper]
            break

    # Strip _orig_mod. prefix from torch.compile
    state_dict = _strip_prefix(state_dict, "_orig_mod.")

    # Detect architecture and build model
    arch_cfg = _detect_config(state_dict)
    model = TalkieModel(arch_cfg)

    # Load weights
    model_sd = model.state_dict()
    ckpt_keys = set(state_dict.keys())
    model_keys = set(model_sd.keys())

    matched = ckpt_keys & model_keys
    missing_in_ckpt = model_keys - ckpt_keys
    extra_in_ckpt = ckpt_keys - model_keys

    for k in matched:
        if model_sd[k].shape == state_dict[k].shape:
            model_sd[k] = state_dict[k]
        else:
            print(f"  Shape mismatch: {k}  model={model_sd[k].shape}  ckpt={state_dict[k].shape}")

    model.load_state_dict(model_sd, strict=False)
    del raw, state_dict

    print(f"[model_loader] Loaded {len(matched)}/{len(model_keys)} params")
    if missing_in_ckpt:
        # Filter out non-persistent buffers (rotary caches)
        real_missing = {k for k in missing_in_ckpt if "rotary" not in k and "cached" not in k}
        if real_missing:
            print(f"  Missing in ckpt: {sorted(real_missing)[:5]}")
    if extra_in_ckpt:
        print(f"  Extra in ckpt:   {sorted(extra_in_ckpt)[:5]}")

    model = model.to(dtype=torch.float32).to(config.DEVICE)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"[model_loader] {n_params/1e9:.2f}B params on {config.DEVICE} (float32)")

    # Tokeniser
    tokeniser = load_tiktoken_tokeniser(repo_dir, expected_vocab=arch_cfg.vocab_size)
    print(f"[model_loader] Tokeniser: vocab={tokeniser.n_vocab}")

    return ModelWrapper(model, tokeniser, model_id)

In [8]:
# Patch model_loader.py on disk
with open("/workspace/Talkie/model_loader.py", "r") as f:
    content = f.read()

content = content.replace('torch.bfloat16', 'torch.float32')
content = content.replace('(bfloat16)', '(float32)')

with open("/workspace/Talkie/model_loader.py", "w") as f:
    f.write(content)

print("Patched!")

# Reload both modules
import importlib
import config
importlib.reload(config)
import model_loader
importlib.reload(model_loader)
from model_loader import load_model

print(f"config.DTYPE = {config.DTYPE}")

Patched!
config.DTYPE = torch.float32


In [9]:
# Write updated data/temporal_facts.json (expanded to 200+ per category)
temporal_facts = json.loads(open(PROJECT_ROOT / 'data' / 'temporal_facts.json').read()) if (PROJECT_ROOT / 'data' / 'temporal_facts.json').exists() else None

# Check if it's already expanded
if temporal_facts and len(temporal_facts.get('pre_1930_true', [])) >= 200:
    print(f"temporal_facts.json already expanded: {len(temporal_facts['pre_1930_true'])} pre_1930_true items")
else:
    print("Writing expanded temporal_facts.json...")
    # The expanded dataset is embedded below
    temporal_facts_expanded = TEMPORAL_FACTS_DATA  # Will be defined in next cell
    (PROJECT_ROOT / 'data').mkdir(exist_ok=True)
    with open(PROJECT_ROOT / 'data' / 'temporal_facts.json', 'w') as f:
        json.dump(temporal_facts_expanded, f, indent=2, ensure_ascii=False)
    print("Written: data/temporal_facts.json")

temporal_facts.json already expanded: 200 pre_1930_true items


In [10]:
# Write the expanded temporal_facts.json directly
import urllib.request

# Since the file is large, write it cell by cell
(PROJECT_ROOT / 'data').mkdir(exist_ok=True)

# Check current size
tf_path = PROJECT_ROOT / 'data' / 'temporal_facts.json'
if tf_path.exists():
    with open(tf_path) as f:
        current = json.load(f)
    sizes = {k: len(v) for k, v in current.items()}
    print(f"Current temporal_facts.json sizes: {sizes}")
    if sizes.get('pre_1930_true', 0) >= 200:
        print("Already expanded! Skipping.")
    else:
        print("Need to write expanded version - see next cell")
else:
    print("File not found - will write in next cell")

Current temporal_facts.json sizes: {'pre_1930_true': 200, 'pre_1930_false': 199, 'post_1930_true': 199, 'post_1930_false': 199, 'leaked_knowledge': 15}
Already expanded! Skipping.


In [12]:
# Restore dataset from write_dataset.py (handles null->None conversion)
import subprocess
result = subprocess.run(['python', 'write_dataset.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

In [13]:
# Verify the temporal_facts.json is correct
with open(PROJECT_ROOT / 'data' / 'temporal_facts.json') as f:
    tf = json.load(f)
print("Temporal facts dataset sizes:")
for k, v in tf.items():
    print(f"  {k}: {len(v)}")
assert len(tf['pre_1930_true']) >= 200, f"Expected 200+ pre_1930_true, got {len(tf['pre_1930_true'])}"
print("\nDataset verification PASSED")

Temporal facts dataset sizes:
  pre_1930_true: 200
  pre_1930_false: 199
  post_1930_true: 199
  post_1930_false: 199
  leaked_knowledge: 15

Dataset verification PASSED


## 1. Mine Leaked Knowledge (~30 min)

Tests ~80 borderline post-1930 facts on Talkie-1930. Facts correctly completed are leaked knowledge candidates.

In [14]:
!pip install tiktoken huggingface_hub datasets scikit-learn numpy pandas matplotlib seaborn scipy tqdm -q


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [16]:
import torch
import importlib
import config
importlib.reload(config)
from model_loader import load_model

print(f"Device: {config.DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
CUDA available: True
GPU: NVIDIA A100 80GB PCIe
Memory: 85.1 GB


In [17]:
!nvidia-smi

Sun May 17 01:10:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:41:00.0 Off |                    0 |
| N/A   37C    P0             46W /  300W |       4MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [18]:
# Post-1930 facts to test, focusing on 1930-1945 borderline period
BORDERLINE_FACTS = [
    # 1930-1935 (most likely to leak through borderline dates)
    {"prompt": "The planet Pluto was discovered in the year", "expected": "1930", "year": 1930},
    {"prompt": "The Empire State Building was completed in", "expected": "1931", "year": 1931},
    {"prompt": "Franklin Roosevelt was first elected president in", "expected": "1932", "year": 1932},
    {"prompt": "Adolf Hitler became Chancellor of Germany in", "expected": "1933", "year": 1933},
    {"prompt": "The Dust Bowl devastated the American", "expected": "Great Plains", "year": 1934},
    {"prompt": "The Hoover Dam was completed in the year", "expected": "1935", "year": 1935},
    {"prompt": "Jesse Owens won four gold medals at the Olympics in", "expected": "Berlin", "year": 1936},
    {"prompt": "The Hindenburg disaster occurred in", "expected": "1937", "year": 1937},
    {"prompt": "Nylon was first commercially produced by", "expected": "DuPont", "year": 1938},
    {"prompt": "Germany invaded Poland in September", "expected": "1939", "year": 1939},
    {"prompt": "The Golden Gate Bridge is located in", "expected": "San Francisco", "year": 1937},
    {"prompt": "The New Deal was a series of programs by President", "expected": "Roosevelt", "year": 1933},
    {"prompt": "Amelia Earhart disappeared over the Pacific in", "expected": "1937", "year": 1937},
    {"prompt": "The Spanish Civil War began in", "expected": "1936", "year": 1936},
    {"prompt": "Penicillin was first used as a medicine by Alexander", "expected": "Fleming", "year": 1929},
    {"prompt": "The stock market crashed in October", "expected": "1929", "year": 1929},
    {"prompt": "The Great Depression began in the year", "expected": "1929", "year": 1929},
    {"prompt": "Mahatma Gandhi led the Salt March in", "expected": "1930", "year": 1930},
    {"prompt": "The Nazis held the Nuremberg rallies in the city of", "expected": "Nuremberg", "year": 1933},
    {"prompt": "Japan invaded Manchuria in", "expected": "1931", "year": 1931},

    # 1935-1945
    {"prompt": "World War II began in the year", "expected": "1939", "year": 1939},
    {"prompt": "The Battle of Britain was fought in", "expected": "1940", "year": 1940},
    {"prompt": "Japan attacked Pearl Harbor on December 7,", "expected": "1941", "year": 1941},
    {"prompt": "The Battle of Stalingrad took place in", "expected": "1942", "year": 1942},
    {"prompt": "D-Day, the Allied invasion of Normandy, occurred in", "expected": "1944", "year": 1944},
    {"prompt": "The United Nations was founded in", "expected": "1945", "year": 1945},
    {"prompt": "The atomic bomb was first tested at", "expected": "Trinity", "year": 1945},
    {"prompt": "Anne Frank wrote her diary while hiding in", "expected": "Amsterdam", "year": 1942},
    {"prompt": "The Manhattan Project developed the first", "expected": "atomic", "year": 1942},
    {"prompt": "Radar technology was crucial in the Battle of", "expected": "Britain", "year": 1940},

    # 1945-1960 (less likely to leak)
    {"prompt": "The Marshall Plan helped rebuild", "expected": "Europe", "year": 1948},
    {"prompt": "The State of Israel was established in", "expected": "1948", "year": 1948},
    {"prompt": "NATO was established in the year", "expected": "1949", "year": 1949},
    {"prompt": "The Korean War began in", "expected": "1950", "year": 1950},
    {"prompt": "Queen Elizabeth II became queen in", "expected": "1952", "year": 1952},
    {"prompt": "The double helix structure of DNA was discovered in", "expected": "1953", "year": 1953},
    {"prompt": "Rosa Parks refused to give up her seat in", "expected": "1955", "year": 1955},
    {"prompt": "Sputnik, the first artificial satellite, was launched by the", "expected": "Soviet", "year": 1957},
    {"prompt": "The European Economic Community was established in", "expected": "1957", "year": 1957},
    {"prompt": "Fidel Castro came to power in Cuba in", "expected": "1959", "year": 1959},

    # 1960-1990 (negative controls)
    {"prompt": "The Cuban Missile Crisis occurred in", "expected": "1962", "year": 1962},
    {"prompt": "John F. Kennedy was assassinated in", "expected": "1963", "year": 1963},
    {"prompt": "The Civil Rights Act was signed in", "expected": "1964", "year": 1964},
    {"prompt": "The first heart transplant was performed by", "expected": "Barnard", "year": 1967},
    {"prompt": "Woodstock music festival took place in", "expected": "1969", "year": 1969},
    {"prompt": "The Watergate scandal involved President", "expected": "Nixon", "year": 1972},
    {"prompt": "The Vietnam War ended in", "expected": "1975", "year": 1975},
    {"prompt": "The Camp David Accords were signed by", "expected": "Carter", "year": 1978},
    {"prompt": "Margaret Thatcher became Prime Minister in", "expected": "1979", "year": 1979},
    {"prompt": "The AIDS epidemic was first identified in", "expected": "1981", "year": 1981},
    {"prompt": "The Falklands War was between Britain and", "expected": "Argentina", "year": 1982},
    {"prompt": "The Internet was originally developed by", "expected": "DARPA", "year": 1969},
    {"prompt": "The first personal computer was the", "expected": "Apple", "year": 1977},
    {"prompt": "Mikhail Gorbachev introduced the policy of", "expected": "perestroika", "year": 1986},
    {"prompt": "Nelson Mandela was released from prison in", "expected": "1990", "year": 1990},

    # Additional borderline 1928-1932
    {"prompt": "Alexander Fleming discovered penicillin in", "expected": "1928", "year": 1928},
    {"prompt": "The first Academy Awards ceremony was held in", "expected": "1929", "year": 1929},
    {"prompt": "The Chrysler Building was completed in New York in", "expected": "1930", "year": 1930},
    {"prompt": "The Star-Spangled Banner became the national anthem in", "expected": "1931", "year": 1931},
    {"prompt": "Aldous Huxley published Brave New World in", "expected": "1932", "year": 1932},
    {"prompt": "The first FIFA World Cup was held in", "expected": "Uruguay", "year": 1930},
    {"prompt": "The Smoot-Hawley Tariff Act was signed in", "expected": "1930", "year": 1930},

    # Science near boundary
    {"prompt": "Edwin Hubble showed that the universe is", "expected": "expanding", "year": 1929},
    {"prompt": "The neutron was discovered by James", "expected": "Chadwick", "year": 1932},
    {"prompt": "Dirac predicted the existence of the", "expected": "positron", "year": 1931},
    {"prompt": "Kurt Godel published his incompleteness theorems in", "expected": "1931", "year": 1931},
    {"prompt": "Heavy water was discovered in", "expected": "1932", "year": 1932},

    # Culture near boundary
    {"prompt": "The first talking motion picture was The Jazz", "expected": "Singer", "year": 1927},
    {"prompt": "Mickey Mouse first appeared in the cartoon Steamboat", "expected": "Willie", "year": 1928},
    {"prompt": "Gone with the Wind was published by Margaret", "expected": "Mitchell", "year": 1936},
    {"prompt": "Walt Disney released the first full-length animated film Snow White in", "expected": "1937", "year": 1937},
]

print(f"Total borderline facts to test: {len(BORDERLINE_FACTS)}")

Total borderline facts to test: 71


In [19]:
# Run the mining
print("=" * 60)
print("MINING FOR LEAKED KNOWLEDGE IN TALKIE-1930")
print("=" * 60)

model = load_model(config.VINTAGE_MODEL_ID)

results = []
correct_by_decade = {}

for item in BORDERLINE_FACTS:
    completion = model.generate(
        item["prompt"], max_new_tokens=30, temperature=0.0
    )
    generated = completion.strip()
    hit = item["expected"].lower() in generated.lower()
    mark = "Y" if hit else "N"

    decade = (item["year"] // 10) * 10
    if decade not in correct_by_decade:
        correct_by_decade[decade] = {"correct": 0, "total": 0}
    correct_by_decade[decade]["total"] += 1
    if hit:
        correct_by_decade[decade]["correct"] += 1

    result = {
        "prompt": item["prompt"],
        "expected": item["expected"],
        "year": item["year"],
        "completion": generated[:100],
        "correct": hit,
    }
    results.append(result)
    print(f"  [{mark}] ({item['year']}) \"{item['prompt']}\"")
    print(f"       -> \"{generated[:70]}\"")

del model
torch.cuda.empty_cache()

# Summary
print(f"\n{'='*60}")
print("SUMMARY BY DECADE")
print(f"{'='*60}")
for decade in sorted(correct_by_decade.keys()):
    d = correct_by_decade[decade]
    pct = d["correct"] / d["total"] * 100 if d["total"] > 0 else 0
    print(f"  {decade}s: {d['correct']}/{d['total']} ({pct:.0f}%)")

leaked_candidates = [r for r in results if r["correct"]]
print(f"\n  Total leaked candidates: {len(leaked_candidates)}")
print(f"  Total tested: {len(results)}")

# Save
config.RESULTS_DIR.mkdir(exist_ok=True)
out_path = config.RESULTS_DIR / "leaked_candidates.json"
with open(out_path, "w") as f:
    json.dump({
        "all_results": results,
        "leaked_candidates": leaked_candidates,
        "by_decade": {str(k): v for k, v in correct_by_decade.items()},
    }, f, indent=2, ensure_ascii=False)
print(f"\n  Saved to {out_path}")

MINING FOR LEAKED KNOWLEDGE IN TALKIE-1930
[model_loader] Using cached repo: /workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base
[model_loader] Checkpoint: final.ckpt (53.1 GB)
[model_loader] Loading checkpoint into RAM ...
[detect] n_layer=40  n_embd=5120  n_head=40  head_dim=128  intermediate=13696  vocab=65536
[model_loader] Loaded 443/443 params
[model_loader] 13.28B params on cuda (float32)
[tokeniser] Loaded 65536 tokens (max rank 65535)
[model_loader] Tokeniser: vocab=65537
  [N] (1930) "The planet Pluto was discovered in the year"
       -> "1892 by M. Charlois, of Nice, France. It is a very small planet, being"
  [N] (1931) "The Empire State Building was completed in"
       -> "1892, and is one of the most imposing structures in the city. It is bu"
  [N] (1932) "Franklin Roosevelt was first elected president in"
       -> "1924. He was reelected in 1928. He was born in New York City in 1879. "
  [Y] (1933) "Adolf Hitler became Chancellor of Germany in"
       -> "19

## 2. Expand Dataset with Leaked Candidates (~1 min)

In [20]:
# Load current dataset and add leaked candidates
data_path = config.DATA_DIR / "temporal_facts.json"
with open(data_path) as f:
    data = json.load(f)

print(f"Current dataset sizes:")
for key, items in data.items():
    print(f"  {key}: {len(items)}")

# Add leaked candidates
existing_texts = {item["text"] for item in data["leaked_knowledge"]}
added = 0
for c in leaked_candidates:
    text = f"{c['prompt']} {c['expected']}"
    if text not in existing_texts:
        data["leaked_knowledge"].append({
            "text": text,
            "year": c["year"],
            "source": "mined",
            "domain": "general",
        })
        existing_texts.add(text)
        added += 1

print(f"\n  Added {added} new leaked items (total: {len(data['leaked_knowledge'])})")

# Save updated dataset
with open(data_path, "w") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)
print(f"  Updated dataset saved to {data_path}")

print(f"\nFinal dataset sizes:")
for key, items in data.items():
    print(f"  {key}: {len(items)}")

Current dataset sizes:
  pre_1930_true: 200
  pre_1930_false: 199
  post_1930_true: 199
  post_1930_false: 199
  leaked_knowledge: 15

  Added 7 new leaked items (total: 22)
  Updated dataset saved to /workspace/Talkie/data/temporal_facts.json

Final dataset sizes:
  pre_1930_true: 200
  pre_1930_false: 199
  post_1930_true: 199
  post_1930_false: 199
  leaked_knowledge: 22


## 3. Experiment 3: Probing with Pipeline Fix (~2.5 hours)

Re-runs probing with:
- StandardScaler inside Pipeline (no train/test leakage)
- Expanded dataset (200+ items per category)
- Updated leaked class (30-50 items from mining)

In [21]:
# Reload config and modules to pick up changes
import importlib
import config
importlib.reload(config)

from experiment3_probing import run_experiment3

print("Running Experiment 3 (Probing) with Pipeline fix + expanded data...")
print(f"Probe layers: {config.PROBE_LAYERS}")
print(f"CV folds: {config.PROBE_CV_FOLDS}")

probing_results = run_experiment3()
print("\nExperiment 3 COMPLETE")

Running Experiment 3 (Probing) with Pipeline fix + expanded data...
Probe layers: [0, 4, 8, 12, 16, 20, 24, 28, 32, 36, 40]
CV folds: 5

Probing: Talkie-1930
[model_loader] Using cached repo: /workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base
[model_loader] Checkpoint: final.ckpt (53.1 GB)
[model_loader] Loading checkpoint into RAM ...
[detect] n_layer=40  n_embd=5120  n_head=40  head_dim=128  intermediate=13696  vocab=65536
[model_loader] Loaded 443/443 params
[model_loader] 13.28B params on cuda (float32)
[tokeniser] Loaded 65536 tokens (max rank 65535)
[model_loader] Tokeniser: vocab=65537

  probe_a_veracity (n=797, classes=2)
    Extracting hidden states...


    Extracting hidden states: 100%|██████████| 797/797 [00:57<00:00, 13.87it/s]


    Running probes...
      Layer  0: 0.645 ± 0.028
      Layer  4: 0.671 ± 0.032
      Layer  8: 0.689 ± 0.032
      Layer 12: 0.665 ± 0.032
      Layer 16: 0.688 ± 0.021
      Layer 20: 0.666 ± 0.029
      Layer 24: 0.701 ± 0.038
      Layer 28: 0.666 ± 0.033
      Layer 32: 0.670 ± 0.021
      Layer 36: 0.635 ± 0.026
      Layer 40: 0.629 ± 0.019

  probe_b_temporal (n=399, classes=2)
    Extracting hidden states...


    Extracting hidden states: 100%|██████████| 399/399 [00:28<00:00, 13.79it/s]


    Running probes...
      Layer  0: 0.945 ± 0.028
      Layer  4: 0.945 ± 0.032
      Layer  8: 0.942 ± 0.025
      Layer 12: 0.955 ± 0.022
      Layer 16: 0.982 ± 0.013
      Layer 20: 0.982 ± 0.010
      Layer 24: 0.980 ± 0.017
      Layer 28: 0.957 ± 0.028
      Layer 32: 0.957 ± 0.023
      Layer 36: 0.950 ± 0.021
      Layer 40: 0.942 ± 0.028

  probe_c_knowledge_boundary (n=421, classes=3)
    Extracting hidden states...


...

    Running probes...
      Layer  0: 0.924 ± 0.030
      Layer  4: 0.919 ± 0.030
      Layer  8: 0.917 ± 0.020
      Layer 12: 0.922 ± 0.022
      Layer 16: 0.960 ± 0.014
      Layer 20: 0.957 ± 0.021
      Layer 24: 0.955 ± 0.019
      Layer 28: 0.934 ± 0.032
      Layer 32: 0.936 ± 0.027
      Layer 36: 0.926 ± 0.028
      Layer 40: 0.929 ± 0.038

Results saved to /workspace/Talkie/results/experiment3_probing.json

Probing: Talkie-Web
[model_loader] Using cached repo: /workspace/Talkie/model_cache/talkie-lm--talkie-web-13b-base
[model_loader] Checkpoint: base.ckpt (53.1 GB)
[model_loader] Loading checkpoint into RAM ...
[detect] n_layer=40  n_embd=5120  n_head=40  head_dim=128  intermediate=13696  vocab=65536
[model_loader] Loaded 443/443 params
[model_loader] 13.28B params on cuda (float32)
[tokeniser] Loaded 65536 tokens (max rank 65535)
[model_loader] Tokeniser: vocab=65537

  probe_a_veracity (n=797, classes=2)
    Extracting hidden states...


    Extracting hidden states: 100%|██████████| 797/797 [00:57<00:00, 13.94it/s]


    Running probes...
      Layer  0: 0.634 ± 0.031
      Layer  4: 0.632 ± 0.035
      Layer  8: 0.703 ± 0.032
      Layer 12: 0.679 ± 0.058
      Layer 16: 0.716 ± 0.041
      Layer 20: 0.784 ± 0.029
      Layer 24: 0.822 ± 0.032
      Layer 28: 0.808 ± 0.034
      Layer 32: 0.750 ± 0.026
      Layer 36: 0.685 ± 0.030
      Layer 40: 0.765 ± 0.018

  probe_b_temporal (n=399, classes=2)
    Extracting hidden states...


    Extracting hidden states: 100%|██████████| 399/399 [00:28<00:00, 13.86it/s]


    Running probes...
      Layer  0: 0.960 ± 0.022
      Layer  4: 0.967 ± 0.023
      Layer  8: 0.972 ± 0.015
      Layer 12: 0.980 ± 0.006
      Layer 16: 0.980 ± 0.017
      Layer 20: 0.995 ± 0.006
      Layer 24: 0.990 ± 0.005
      Layer 28: 0.988 ± 0.008
      Layer 32: 0.993 ± 0.006
      Layer 36: 0.985 ± 0.015
      Layer 40: 0.980 ± 0.013

Results saved to /workspace/Talkie/results/experiment3_probing.json

Experiment 3 COMPLETE


## 4. Extended Probe Analysis: MLP + Permutation + Bootstrap (~30 min)

In [22]:
import importlib
import config
importlib.reload(config)

from probe_analysis import (
    probe_c_detailed_metrics,
    lexical_baselines,
    bootstrap_confidence_intervals,
    mlp_probe_robustness,
    permutation_baseline,
    behavioural_knowledge_test,
)

all_results = {}

print("Running extended probe analysis...")

# 1. Probe C detailed metrics
probe_c_results = probe_c_detailed_metrics()
if probe_c_results:
    all_results["probe_c_detailed"] = {
        str(k): {
            "per_class_f1": v["per_class_f1"],
            "confusion_matrix": v["confusion_matrix"],
        } for k, v in probe_c_results.items()
    }

Running extended probe analysis...
1. PROBE C — Per-Class Metrics & Confusion Matrix

  Layer 16:
                             precision    recall  f1-score   support

     should-know (pre-1930)       0.97      0.99      0.98       200
should-not-know (post-1930)       0.95      0.98      0.97       199
                     leaked       1.00      0.45      0.62        22

                   accuracy                           0.96       421
                  macro avg       0.97      0.81      0.86       421
               weighted avg       0.96      0.96      0.95       421

  Confusion Matrix:
                                 Predicted
                                  should-know  shouldnt-know   leaked
        should-know (pre-1930)       198         2         0
   should-not-know (post-1930)         3       196         0
                        leaked         4         8        10

  Layer 20:
                             precision    recall  f1-score   support

     should-know 

In [23]:
# 2. Lexical baselines
lexical_results = lexical_baselines()
if lexical_results:
    all_results["lexical_baselines"] = lexical_results


2. LEXICAL BASELINES (TF-IDF Bag-of-Words)

  probe_a_veracity (n=797, classes=2)
    TF-IDF baseline: 0.472 +/- 0.022
    Talkie-1930 best neural probe (layer 24): 0.701  (delta = +0.230)
    Talkie-Web best neural probe (layer 24): 0.822  (delta = +0.350)

  probe_b_temporal (n=399, classes=2)
    TF-IDF baseline: 0.774 +/- 0.029
    Talkie-1930 best neural probe (layer 16): 0.982  (delta = +0.208)
    Talkie-Web best neural probe (layer 20): 0.995  (delta = +0.221)

  probe_c_knowledge_boundary (n=421, classes=3)
    TF-IDF baseline: 0.751 +/- 0.032
    Talkie-1930 best neural probe (layer 16): 0.960  (delta = +0.209)


In [24]:
# 3. Bootstrap CIs
ci_results = bootstrap_confidence_intervals()
if ci_results:
    all_results["bootstrap_cis"] = ci_results


3. BOOTSTRAP CONFIDENCE INTERVALS

  BLiMP aggregate CIs:
    Talkie-1930: 0.793 [0.739, 0.850] 95% CI
    Talkie-Web: 0.839 [0.796, 0.886] 95% CI
    Gap (Modern - Vintage): 0.046 [0.029, 0.064] p=0.0000

  ICL slope CIs (from seed variation):
    Talkie-1930 mean slope: 0.0495 [0.0206, 0.0785]
    Talkie-Web mean slope: 0.0481 [0.0184, 0.0777]


In [25]:
# 4. MLP probe robustness
mlp_results = mlp_probe_robustness()
if mlp_results:
    all_results["mlp_robustness"] = mlp_results


5. MLP PROBE ROBUSTNESS CHECK

  probe_a_veracity (n=797, classes=2)
    Talkie-1930:
      Linear (layer 24): 0.701
      MLP    (layer 16): 0.714
      Delta (MLP - Linear): +0.013
    Talkie-Web:
      Linear (layer 24): 0.822
      MLP    (layer 20): 0.754
      Delta (MLP - Linear): -0.068

  probe_b_temporal (n=399, classes=2)
    Talkie-1930:
      Linear (layer 16): 0.982
      MLP    (layer 20): 0.970
      Delta (MLP - Linear): -0.013
    Talkie-Web:
      Linear (layer 20): 0.995
      MLP    (layer 32): 0.970
      Delta (MLP - Linear): -0.025

  probe_c_knowledge_boundary (n=421, classes=3)
    Talkie-1930:
      Linear (layer 16): 0.960
      MLP    (layer 20): 0.936
      Delta (MLP - Linear): -0.024


In [26]:
# 5. Permutation baseline
perm_results = permutation_baseline()
if perm_results:
    all_results["permutation_baseline"] = perm_results


6. PERMUTATION BASELINE (SHUFFLED LABELS)

  probe_a_veracity (n=797, classes=2)
    Talkie-1930 (layer 24):
      Real accuracy:       0.701
      Permuted accuracy:   0.501 +/- 0.024
      Permutation p-value: 0.0000
    Talkie-Web (layer 24):
      Real accuracy:       0.822
      Permuted accuracy:   0.499 +/- 0.024
      Permutation p-value: 0.0000

  probe_b_temporal (n=399, classes=2)
    Talkie-1930 (layer 16):
      Real accuracy:       0.982
      Permuted accuracy:   0.500 +/- 0.030
      Permutation p-value: 0.0000
    Talkie-Web (layer 20):
      Real accuracy:       0.995
      Permuted accuracy:   0.495 +/- 0.027
      Permutation p-value: 0.0000

  probe_c_knowledge_boundary (n=421, classes=3)
    Talkie-1930 (layer 16):
      Real accuracy:       0.960
      Permuted accuracy:   0.476 +/- 0.029
      Permutation p-value: 0.0000


In [ ]:
## yahan tak sab thik hai

In [27]:
# 6. Behavioural knowledge test (requires GPU)
behavioural_knowledge_test()


4. BEHAVIOURAL KNOWLEDGE BOUNDARY TEST
  (Requires GPU — run on RunPod)
  Test definition saved to /workspace/Talkie/results/knowledge_boundary_test.json

  Testing: Talkie-1930
[model_loader] Using cached repo: /workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base
[model_loader] Checkpoint: final.ckpt (53.1 GB)
[model_loader] Loading checkpoint into RAM ...
[detect] n_layer=40  n_embd=5120  n_head=40  head_dim=128  intermediate=13696  vocab=65536
[model_loader] Loaded 443/443 params
[model_loader] 13.28B params on cuda (float32)
[tokeniser] Loaded 65536 tokens (max rank 65535)
[model_loader] Tokeniser: vocab=65537
    [N] "The inventor of the telephone was" -> "a poor man, and he had to work hard to get the money to pay "
    [Y] "The Titanic sank in the year" -> "1912, and the Lusitania in 1915. The Titanic was the largest"
    [Y] "The theory of relativity was proposed by" -> "Einstein in 1905, and has been developed by him and others s"
    [N] "The first modern Olympic Ga

In [28]:
# Save all extended analysis results
out_path = config.RESULTS_DIR / "probe_analysis_extended.json"
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2, default=str)
print(f"\nExtended analysis saved to {out_path}")
print("\nSection 4 COMPLETE")


Extended analysis saved to /workspace/Talkie/results/probe_analysis_extended.json

Section 4 COMPLETE


## 5. Qualitative Generations (~10 min)

In [29]:
import importlib
import config
importlib.reload(config)
from model_loader import load_model

PROMPTS = [
    # Pre-1930 facts (Vintage SHOULD know)
    {"prompt": "The inventor of the telephone was", "expected": "Alexander Graham Bell", "period": "pre-1930"},
    {"prompt": "The Titanic sank in the year", "expected": "1912", "period": "pre-1930"},
    {"prompt": "The theory of relativity was proposed by", "expected": "Einstein", "period": "pre-1930"},
    {"prompt": "The Eiffel Tower is located in", "expected": "Paris", "period": "pre-1930"},
    {"prompt": "Marie Curie discovered the element", "expected": "radium", "period": "pre-1930"},
    {"prompt": "World War I began in the year", "expected": "1914", "period": "pre-1930"},

    # Post-1930 facts (Vintage SHOULD NOT know)
    {"prompt": "The first person to walk on the moon was", "expected": "Neil Armstrong", "period": "post-1930"},
    {"prompt": "The Berlin Wall fell in the year", "expected": "1989", "period": "post-1930"},
    {"prompt": "The structure of DNA was discovered by Watson and", "expected": "Crick", "period": "post-1930"},
    {"prompt": "The first atomic bomb was dropped on", "expected": "Hiroshima", "period": "post-1930"},
    {"prompt": "The World Wide Web was invented by", "expected": "Tim Berners-Lee", "period": "post-1930"},
    {"prompt": "The first iPhone was released in", "expected": "2007", "period": "post-1930"},
    {"prompt": "The Chernobyl nuclear disaster occurred in", "expected": "1986", "period": "post-1930"},
    {"prompt": "The Soviet Union collapsed in", "expected": "1991", "period": "post-1930"},
]

all_gen_results = {}

for model_id in [config.VINTAGE_MODEL_ID, config.MODERN_MODEL_ID]:
    model_name = config.MODEL_NAMES.get(model_id, model_id)
    print(f"\n{'='*60}")
    print(f"Generating: {model_name}")
    print(f"{'='*60}")

    model = load_model(model_id)
    model_results = []

    for item in PROMPTS:
        completion = model.generate(
            item["prompt"], max_new_tokens=30, temperature=0.0
        )
        generated = completion.strip()
        hit = item["expected"].lower() in generated.lower()
        mark = "Y" if hit else "N"

        result = {
            "prompt": item["prompt"],
            "expected": item["expected"],
            "period": item["period"],
            "completion": generated,
            "correct": hit,
        }
        model_results.append(result)
        print(f"  [{mark}] \"{item['prompt']}\"")
        print(f"       -> \"{generated[:80]}\"")

    all_gen_results[model_name] = model_results

    del model
    torch.cuda.empty_cache()

out_path = config.RESULTS_DIR / "qualitative_generations.json"
with open(out_path, "w") as f:
    json.dump(all_gen_results, f, indent=2, ensure_ascii=False)
print(f"\nSaved to {out_path}")
print("\nSection 5 COMPLETE")


Generating: Talkie-1930
[model_loader] Using cached repo: /workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base
[model_loader] Checkpoint: final.ckpt (53.1 GB)
[model_loader] Loading checkpoint into RAM ...
[detect] n_layer=40  n_embd=5120  n_head=40  head_dim=128  intermediate=13696  vocab=65536
[model_loader] Loaded 443/443 params
[model_loader] 13.28B params on cuda (float32)
[tokeniser] Loaded 65536 tokens (max rank 65535)
[model_loader] Tokeniser: vocab=65537
  [N] "The inventor of the telephone was"
       -> "a poor man, and he had to work hard to get the money to pay for the patent. He h"
  [Y] "The Titanic sank in the year"
       -> "1912, and the Lusitania in 1915. The Titanic was the largest ship afloat at the "
  [Y] "The theory of relativity was proposed by"
       -> "Einstein in 1905, and has been developed by him and others since that time. It i"
  [N] "The Eiffel Tower is located in"
       -> "the Champ de Mars, and is the most conspicuous object in the Expo

## 6. ICL Experiment with 10 Seeds (~2.5 hours)

Re-runs ICL with 10 seeds instead of 3 for tighter confidence intervals.

In [6]:
import sys
sys.path.insert(0, '/workspace/Talkie')
import config
import time

config.ICL_K_VALUES = [0, 1, 2, 4, 8, 16, 32]
config.ICL_SEEDS = [42, 123, 456, 789, 101, 202, 303, 404, 505, 606]
config.ICL_MAX_EVAL_SAMPLES = 500

import experiment2_icl
from experiment2_icl import run_experiment2, evaluate_task

# Patch evaluate_task for progress monitoring
_orig_evaluate = evaluate_task
_run_count = [0]
_total_runs = len(config.ICL_SEEDS) * len(config.ICL_K_VALUES) * len(config.ICL_TASKS) * 2
_start_time = [None]

def patched_evaluate(model, task_name, demo_pool, eval_set, k, seed):
    if _start_time[0] is None:
        _start_time[0] = time.time()
    _run_count[0] += 1
    elapsed = time.time() - _start_time[0]
    pct = _run_count[0] / _total_runs * 100
    rate = _run_count[0] / max(elapsed, 1)
    remaining = (_total_runs - _run_count[0]) / max(rate, 0.001)
    remaining_h = remaining / 3600
    print(f"      [{_run_count[0]}/{_total_runs} ({pct:.0f}%)] k={k}, seed={seed} | "
          f"elapsed: {elapsed/3600:.1f}h, ETA: {remaining_h:.1f}h", flush=True)
    result = _orig_evaluate(model, task_name, demo_pool, eval_set, k, seed)
    print(f"        -> acc={result['accuracy']:.3f}", flush=True)
    return result

experiment2_icl.evaluate_task = patched_evaluate

print(f"Starting {_total_runs} runs (10 seeds x 7 k-values x 4 tasks x 2 models)")
icl_results = run_experiment2()
print("\nExperiment 2 COMPLETE (10 seeds)")

Starting 560 runs (10 seeds x 7 k-values x 4 tasks x 2 models)
[exp2] Loading sst2...


    Eval set: 436 samples, label dist: {0: 214, 1: 222}
[exp2] Loading mnli...
    Eval set: 500 samples, label dist: {0: 177, 1: 149, 2: 174}
[exp2] Loading tweet_sentiment...
    Eval set: 500 samples, label dist: {0: 154, 1: 237, 2: 109}
[exp2] Loading tweet_emotion...
    Eval set: 500 samples, label dist: {0: 206, 1: 124, 2: 39, 3: 131}

Evaluating: Talkie-1930
[model_loader] Using cached repo: /workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base
[model_loader] Checkpoint: final.ckpt (53.1 GB)
[model_loader] Loading checkpoint into RAM ...
[detect] n_layer=40  n_embd=5120  n_head=40  head_dim=128  intermediate=13696  vocab=65536
[model_loader] Loaded 443/443 params
[model_loader] 13.28B params on cuda (float32)
[tokeniser] Loaded 65536 tokens (max rank 65535)
[model_loader] Tokeniser: vocab=65537

  Task: sst2
      [1/560 (0%)] k=0, seed=42 | elapsed: 0.0h, ETA: 0.2h
        -> acc=0.532
      [2/560 (0%)] k=0, seed=123 | elapsed: 0.0h, ETA: 7.3h
        -> acc=0.532
   

KeyboardInterrupt: 

In [1]:
# Cell 1: Setup
import importlib
import config
importlib.reload(config)

config.ICL_K_VALUES = [0, 1, 4, 8, 16]
config.ICL_SEEDS = [42, 123, 456]
config.ICL_MAX_EVAL_SAMPLES = 200

import experiment2_icl
importlib.reload(experiment2_icl)
from experiment2_icl import run_experiment2

icl_results = run_experiment2()
print("\nExperiment 2 COMPLETE")

[exp2] Loading sst2...


    Eval set: 200 samples, label dist: {0: 103, 1: 97}
[exp2] Loading mnli...
    Eval set: 200 samples, label dist: {0: 69, 1: 58, 2: 73}
[exp2] Loading tweet_sentiment...
    Eval set: 200 samples, label dist: {0: 61, 1: 93, 2: 46}
[exp2] Loading tweet_emotion...
    Eval set: 200 samples, label dist: {0: 82, 1: 58, 2: 15, 3: 45}

Evaluating: Talkie-1930
[model_loader] Using cached repo: /workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base
[model_loader] Checkpoint: final.ckpt (53.1 GB)
[model_loader] Loading checkpoint into RAM ...
[detect] n_layer=40  n_embd=5120  n_head=40  head_dim=128  intermediate=13696  vocab=65536
[model_loader] Loaded 443/443 params
[model_loader] 13.28B params on cuda (float32)
[tokeniser] Loaded 65536 tokens (max rank 65535)
[model_loader] Tokeniser: vocab=65537

  Task: sst2
    k= 0: 0.510 ± 0.000
    k= 1: 0.725 ± 0.061
    k= 4: 0.870 ± 0.007


KeyboardInterrupt: 

In [3]:
import json

with open('/workspace/Talkie/results/experiment2_icl.json') as f:
    d = json.load(f)

# Check if results have 10 seeds (original) or 3 seeds (reduced)
first_task = d['Talkie-1930']['sst2']['1']
n_seeds = len(first_task['per_seed'])
has_k32 = '32' in d['Talkie-1930']['sst2']

print(f"Seeds per result: {n_seeds}")
print(f"Has k=32: {has_k32}")
print(f"K values: {sorted(int(k) for k in d['Talkie-1930']['sst2'] if k.isdigit())}")

Seeds per result: 3
Has k=32: True
K values: [0, 1, 2, 4, 8, 16, 32]


In [4]:
import os
results_dir = '/workspace/Talkie/results/'
for f in sorted(os.listdir(results_dir)):
    size = os.path.getsize(os.path.join(results_dir, f))
    print(f'{f:50s} {size:>10,} bytes')

experiment1_full.json                              51,061,697 bytes
experiment1_syntactic.json                              4,365 bytes
experiment2_icl.json                                   13,517 bytes
experiment3_probing.json                               17,005 bytes
hidden_states_Talkie-1930_probe_a_veracity.npz     166,438,730 bytes
hidden_states_Talkie-1930_probe_b_temporal.npz     83,342,993 bytes
hidden_states_Talkie-1930_probe_c_knowledge_boundary.npz 87,938,124 bytes
hidden_states_Talkie-Web_probe_a_veracity.npz      166,470,016 bytes
hidden_states_Talkie-Web_probe_b_temporal.npz      83,358,763 bytes
knowledge_boundary_Talkie-1930.json                       155 bytes
knowledge_boundary_Talkie-Web.json                        172 bytes
knowledge_boundary_test.json                            3,014 bytes
leaked_candidates.json                                 21,426 bytes
ocr_ablation.json                                       5,447 bytes
probe_analysis_extended.json            

In [30]:
import importlib
import config
importlib.reload(config)

print(f"ICL Seeds: {config.ICL_SEEDS}")
print(f"ICL K values: {config.ICL_K_VALUES}")
print(f"ICL Tasks: {list(config.ICL_TASKS.keys())}")
print(f"Max eval samples: {config.ICL_MAX_EVAL_SAMPLES}")
print(f"\nTotal runs: {len(config.ICL_SEEDS)} seeds x {len(config.ICL_K_VALUES)} k-values x {len(config.ICL_TASKS)} tasks x 2 models")
print(f"         = {len(config.ICL_SEEDS) * len(config.ICL_K_VALUES) * len(config.ICL_TASKS) * 2} experiment runs")

ICL Seeds: [42, 123, 456, 789, 101, 202, 303, 404, 505, 606]
ICL K values: [0, 1, 2, 4, 8, 16, 32]
ICL Tasks: ['sst2', 'mnli', 'tweet_sentiment', 'tweet_emotion']
Max eval samples: 500

Total runs: 10 seeds x 7 k-values x 4 tasks x 2 models
         = 560 experiment runs


In [ ]:
from experiment2_icl import run_experiment2

print("Running Experiment 2 (ICL) with 10 seeds...")
print("This will take approximately 2.5 hours.")

icl_results = run_experiment2()
print("\nExperiment 2 COMPLETE")


In [8]:
# checking the results 
import json

with open("/workspace/Talkie/results/experiment2_icl.json", "r") as f:
    data = json.load(f)

for model_name, model_data in data.items():
    print(f"\n{model_name}:")
    for task_name, task_data in model_data.items():
        k_values = sorted([k for k in task_data.keys() if k.isdigit()], key=int)
        seeds_per_k = [len(task_data[k].get("per_seed", [])) for k in k_values]
        print(f"  {task_name}: k={k_values}, seeds={seeds_per_k}")


Talkie-1930:
  sst2: k=['0', '1', '2', '4', '8', '16', '32'], seeds=[3, 3, 3, 3, 3, 3, 3]
  mnli: k=['0', '1', '2', '4', '8', '16', '32'], seeds=[3, 3, 3, 3, 3, 3, 3]
  tweet_sentiment: k=['0', '1', '2', '4', '8', '16', '32'], seeds=[3, 3, 3, 3, 3, 3, 3]
  tweet_emotion: k=['0', '1', '2', '4', '8', '16', '32'], seeds=[3, 3, 3, 3, 3, 3, 3]

Talkie-Web:
  sst2: k=['0', '1', '2', '4', '8', '16', '32'], seeds=[3, 3, 3, 3, 3, 3, 3]
  mnli: k=['0', '1', '2', '4', '8', '16', '32'], seeds=[3, 3, 3, 3, 3, 3, 3]
  tweet_sentiment: k=['0', '1', '2', '4', '8', '16', '32'], seeds=[3, 3, 3, 3, 3, 3, 3]
  tweet_emotion: k=['0', '1', '2', '4', '8', '16', '32'], seeds=[3, 3, 3, 3, 3, 3, 3]


In [10]:
#save the results
import json
import numpy as np

with open("/workspace/Talkie/results/experiment2_icl.json", "r") as f:
    data = json.load(f)

# ============================================================
# Talkie-1930 SST-2 — 10 seeds (reconstructed from console)
# Seeds: 42, 123, 456, 789, 101, 202, 303, 404, 505, 606
# ============================================================
sst2 = data["Talkie-1930"]["sst2"]

sst2["0"]["per_seed"] = [0.532]*10
sst2["1"]["per_seed"] = [0.812, 0.658, 0.794, 0.750, 0.612, 0.794, 0.640, 0.649, 0.667, 0.796]
sst2["2"]["per_seed"] = [0.523, 0.592, 0.869, 0.569, 0.672, 0.872, 0.628, 0.560, 0.846, 0.851]
sst2["4"]["per_seed"] = [0.860, 0.874, 0.881, 0.885, 0.839, 0.814, 0.677, 0.817, 0.830, 0.892]
sst2["8"]["per_seed"] = [0.908, 0.872, 0.661, 0.878, 0.904, 0.885, 0.885, 0.920, 0.890, 0.856]
sst2["16"]["per_seed"] = [0.899, 0.917, 0.917, 0.922, 0.901, 0.853, 0.915, 0.917, 0.892, 0.929]
sst2["32"]["per_seed"] = [0.904, 0.929, 0.922, 0.917, 0.915, 0.917, 0.917, 0.849, 0.936, 0.911]

# ============================================================
# Talkie-1930 MNLI — 10 seeds (reconstructed from console)
# ============================================================
mnli = data["Talkie-1930"]["mnli"]

mnli["0"]["per_seed"] = [0.356]*10
mnli["1"]["per_seed"] = [0.458, 0.498, 0.364, 0.470, 0.480, 0.504, 0.468, 0.450, 0.516, 0.520]
mnli["2"]["per_seed"] = [0.366, 0.448, 0.348, 0.452, 0.348, 0.348, 0.446, 0.366, 0.348, 0.348]
mnli["4"]["per_seed"] = [0.454, 0.412, 0.348, 0.350, 0.384, 0.356, 0.538, 0.430, 0.306, 0.348]
mnli["8"]["per_seed"] = [0.582, 0.422, 0.386, 0.360, 0.412, 0.472, 0.512, 0.422, 0.518, 0.392]
mnli["16"]["per_seed"] = [0.508, 0.422, 0.568, 0.438, 0.538, 0.420, 0.560, 0.446, 0.458, 0.534]
mnli["32"]["per_seed"] = [0.456, 0.354, 0.424, 0.506, 0.362, 0.504, 0.544, 0.456, 0.532, 0.590]

# Update mean/std for all reconstructed entries
for task_data in [sst2, mnli]:
    for k_str in task_data:
        if k_str == "metrics":
            continue
        seeds = task_data[k_str]["per_seed"]
        task_data[k_str]["mean_accuracy"] = float(np.mean(seeds))
        task_data[k_str]["std_accuracy"] = float(np.std(seeds))

# Save 10-seed file (SST-2 & MNLI have 10, rest have 3 for now)
with open("/workspace/Talkie/results/experiment2_icl_10seed_partial.json", "w") as f:
    json.dump(data, f, indent=2)

print("Saved: experiment2_icl_10seed_partial.json")
print("Original experiment2_icl.json untouched (3-seed backup)\n")

# Verify
print("10-seed (reconstructed):")
for task in ["sst2", "mnli"]:
    td = data["Talkie-1930"][task]
    for k in ["0","1","2","4","8","16","32"]:
        n = len(td[k]["per_seed"])
        print(f"  Talkie-1930/{task} k={k}: {td[k]['mean_accuracy']:.3f} ± {td[k]['std_accuracy']:.3f} ({n} seeds)")

print("\nStill 3 seeds (will add 2 more for 5 total):")
for model in ["Talkie-1930", "Talkie-Web"]:
    tasks = ["tweet_sentiment", "tweet_emotion"] if model == "Talkie-1930" else ["sst2", "mnli", "tweet_sentiment", "tweet_emotion"]
    for task in tasks:
        n = len(data[model][task]["0"]["per_seed"])
        print(f"  {model}/{task}: {n} seeds")

Saved: experiment2_icl_10seed_partial.json
Original experiment2_icl.json untouched (3-seed backup)

10-seed (reconstructed):
  Talkie-1930/sst2 k=0: 0.532 ± 0.000 (10 seeds)
  Talkie-1930/sst2 k=1: 0.717 ± 0.075 (10 seeds)
  Talkie-1930/sst2 k=2: 0.698 ± 0.137 (10 seeds)
  Talkie-1930/sst2 k=4: 0.837 ± 0.060 (10 seeds)
  Talkie-1930/sst2 k=8: 0.866 ± 0.071 (10 seeds)
  Talkie-1930/sst2 k=16: 0.906 ± 0.021 (10 seeds)
  Talkie-1930/sst2 k=32: 0.912 ± 0.023 (10 seeds)
  Talkie-1930/mnli k=0: 0.356 ± 0.000 (10 seeds)
  Talkie-1930/mnli k=1: 0.473 ± 0.043 (10 seeds)
  Talkie-1930/mnli k=2: 0.382 ± 0.044 (10 seeds)
  Talkie-1930/mnli k=4: 0.393 ± 0.064 (10 seeds)
  Talkie-1930/mnli k=8: 0.448 ± 0.067 (10 seeds)
  Talkie-1930/mnli k=16: 0.489 ± 0.055 (10 seeds)
  Talkie-1930/mnli k=32: 0.473 ± 0.073 (10 seeds)

Still 3 seeds (will add 2 more for 5 total):
  Talkie-1930/tweet_sentiment: 3 seeds
  Talkie-1930/tweet_emotion: 3 seeds
  Talkie-Web/sst2: 3 seeds
  Talkie-Web/mnli: 3 seeds
  Talkie-

In [1]:
##2-extra-seeds script

In [1]:
# with incremental save 
import sys
sys.path.insert(0, '/workspace/Talkie')
import config
import time
import json
import gc
import torch
import numpy as np

config.ICL_K_VALUES = [0, 1, 2, 4, 8, 16, 32]
config.ICL_SEEDS = [789, 101]
config.ICL_MAX_EVAL_SAMPLES = 500

import experiment2_icl
from experiment2_icl import evaluate_task, load_task_data
from model_loader import load_model

SAVE_PATH = "/workspace/Talkie/results/experiment2_icl_final.json"

# Start from the 10seed_partial file
with open("/workspace/Talkie/results/experiment2_icl_10seed_partial.json", "r") as f:
    final = json.load(f)

# What needs 2 extra seeds
REMAINING = [
    ("talkie-lm/talkie-1930-13b-base", "Talkie-1930", ["tweet_sentiment", "tweet_emotion"]),
    ("talkie-lm/talkie-web-13b-base", "Talkie-Web", ["sst2", "mnli", "tweet_sentiment", "tweet_emotion"]),
]

# Check what's already done (in case of restart)
for _, display_name, tasks in REMAINING:
    for task in tasks[:]:
        n = len(final[display_name][task]["0"]["per_seed"])
        if n >= 5:
            print(f"  SKIP {display_name}/{task} (already {n} seeds)")
            tasks.remove(task)

# Preload task data
task_cache = {}
all_tasks = set()
for _, _, tasks in REMAINING:
    all_tasks.update(tasks)
for t in all_tasks:
    print(f"Loading {t}...")
    task_cache[t] = load_task_data(t)

total_runs = sum(len(tasks) * 7 * 2 for _, _, tasks in REMAINING)
run_count = 0
t0 = time.time()

for model_id, display_name, tasks in REMAINING:
    if not tasks:
        continue
    print(f"\n{'='*60}")
    print(f"Evaluating: {display_name} ({len(tasks)} tasks)")
    print(f"{'='*60}")
    
    model = load_model(model_id)
    
    for task_name in tasks:
        demo_pool, eval_set = task_cache[task_name]
        print(f"\n  Task: {task_name}")
        
        for k in config.ICL_K_VALUES:
            k_str = str(k)
            for seed in config.ICL_SEEDS:
                run_count += 1
                elapsed = time.time() - t0
                eta = (elapsed / run_count) * (total_runs - run_count)
                print(f"    [{run_count}/{total_runs}] k={k}, seed={seed} | "
                      f"elapsed: {elapsed/3600:.1f}h, ETA: {eta/3600:.1f}h", flush=True)
                
                result = evaluate_task(model, task_name, demo_pool, eval_set, k, seed)
                acc = result["accuracy"]
                print(f"      -> acc={acc:.3f}", flush=True)
                final[display_name][task_name][k_str]["per_seed"].append(acc)
            
            # Update stats
            seeds = final[display_name][task_name][k_str]["per_seed"]
            final[display_name][task_name][k_str]["mean_accuracy"] = float(np.mean(seeds))
            final[display_name][task_name][k_str]["std_accuracy"] = float(np.std(seeds))
        
        # SAVE AFTER EACH TASK
        with open(SAVE_PATH, "w") as f:
            json.dump(final, f, indent=2)
        n = len(final[display_name][task_name]["0"]["per_seed"])
        print(f"  [SAVED] {display_name}/{task_name} done ({n} seeds)")
    
    del model
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nDONE! Total time: {(time.time()-t0)/3600:.1f}h")
print("\nFinal seed counts:")
for m in ["Talkie-1930", "Talkie-Web"]:
    for t in ["sst2", "mnli", "tweet_sentiment", "tweet_emotion"]:
        n = len(final[m][t]["0"]["per_seed"])
        print(f"  {m}/{t}: {n} seeds")

Loading tweet_sentiment...


    Eval set: 500 samples, label dist: {0: 154, 1: 237, 2: 109}
Loading tweet_emotion...
    Eval set: 500 samples, label dist: {0: 206, 1: 124, 2: 39, 3: 131}
Loading sst2...
    Eval set: 436 samples, label dist: {0: 214, 1: 222}
Loading mnli...
    Eval set: 500 samples, label dist: {0: 177, 1: 149, 2: 174}

Evaluating: Talkie-1930 (2 tasks)
[model_loader] Using cached repo: /workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base
[model_loader] Checkpoint: final.ckpt (53.1 GB)
[model_loader] Loading checkpoint into RAM ...
[detect] n_layer=40  n_embd=5120  n_head=40  head_dim=128  intermediate=13696  vocab=65536
[model_loader] Loaded 443/443 params
[model_loader] 13.28B params on cuda (float32)
[tokeniser] Loaded 65536 tokens (max rank 65535)
[model_loader] Tokeniser: vocab=65537

  Task: tweet_sentiment
    [1/84] k=0, seed=789 | elapsed: 0.1h, ETA: 4.9h
      -> acc=0.300
    [2/84] k=0, seed=101 | elapsed: 0.1h, ETA: 4.4h
      -> acc=0.300
    [3/84] k=1, seed=789 | elapse

In [9]:
print(data["Talkie-1930"]["sst2"]["1"]["per_seed"])

[0.8027522935779816, 0.6536697247706422, 0.7752293577981652]


In [5]:
import json
with open("/workspace/Talkie/results/experiment2_icl_10seed_partial.json") as f:
    data = json.load(f)
print("JSON keys:", list(data.keys()))

import inspect, sys
sys.path.insert(0, '/workspace/Talkie')
import experiment2_icl
src = inspect.getsource(experiment2_icl.run_experiment2)
for line in src.split('\n')[:25]:
    print(line)
    

JSON keys: ['Talkie-1930', 'Talkie-Web']
def run_experiment2(
    model_ids: list[str] | None = None,
    tasks: list[str] | None = None,
) -> dict:
    """Run full Experiment 2 for all models and tasks."""
    if model_ids is None:
        model_ids = [config.VINTAGE_MODEL_ID, config.MODERN_MODEL_ID]
    if tasks is None:
        tasks = list(config.ICL_TASKS.keys())

    # Pre-load all task data
    task_data = {}
    for task_name in tasks:
        print(f"[exp2] Loading {task_name}...")
        task_data[task_name] = load_task_data(task_name)

    all_results = {}

    for model_id in model_ids:
        model_name = config.MODEL_NAMES.get(model_id, model_id)
        print(f"\n{'='*60}")
        print(f"Evaluating: {model_name}")
        print(f"{'='*60}")

        model = load_model(model_id)


In [6]:
import config
print("VINTAGE:", config.VINTAGE_MODEL_ID)
print("MODERN:", config.MODERN_MODEL_ID)
print("NAMES:", config.MODEL_NAMES)

VINTAGE: talkie-lm/talkie-1930-13b-base
MODERN: talkie-lm/talkie-web-13b-base
NAMES: {'talkie-lm/talkie-1930-13b-base': 'Talkie-1930', 'talkie-lm/talkie-web-13b-base': 'Talkie-Web'}


In [ ]:
## for remaining experimants 5 seed 

In [3]:
import os
print(os.listdir("/workspace/Talkie/model_cache/"))

['talkie-lm--talkie-web-13b-base', 'talkie-lm--talkie-1930-13b-base']


## 7. Regenerate Figures

In [1]:
from pathlib import Path
PROJECT_ROOT = Path('/workspace/Talkie')

In [2]:
# Check if generate_figures.py exists
if (PROJECT_ROOT / 'generate_figures.py').exists():
    exec(open(PROJECT_ROOT / 'generate_figures.py').read())
    print("Figures regenerated!")
else:
    print("generate_figures.py not found - figures will use existing versions")
    print("You can regenerate figures locally from the results JSON files.")

generate_figures.py not found - figures will use existing versions
You can regenerate figures locally from the results JSON files.


## 8. Final Summary & Verification

In [3]:
import sys
sys.path.insert(0, '/workspace/Talkie')
import importlib
import config
importlib.reload(config)
import json

In [4]:
print("=" * 60)
print("ALL EXPERIMENTS COMPLETE")
print("=" * 60)

print("\nResults files:")
for f in sorted(config.RESULTS_DIR.glob("*.json")):
    size = f.stat().st_size / 1024
    print(f"  {f.name:50s} ({size:.1f} KB)")

print("\nFigures:")
for f in sorted(config.FIGURES_DIR.glob("*")):
    size = f.stat().st_size / 1024
    print(f"  {f.name:50s} ({size:.1f} KB)")

print("\nData:")
for f in sorted(config.DATA_DIR.glob("*.json")):
    size = f.stat().st_size / 1024
    print(f"  {f.name:50s} ({size:.1f} KB)")

ALL EXPERIMENTS COMPLETE

Results files:
  experiment1_full.json                              (49864.9 KB)
  experiment1_syntactic.json                         (4.3 KB)
  experiment2_icl.json                               (13.2 KB)
  experiment3_probing.json                           (16.6 KB)
  knowledge_boundary_Talkie-1930.json                (0.2 KB)
  knowledge_boundary_Talkie-Web.json                 (0.2 KB)
  knowledge_boundary_test.json                       (2.9 KB)
  leaked_candidates.json                             (20.9 KB)
  ocr_ablation.json                                  (5.3 KB)
  probe_analysis_extended.json                       (5.4 KB)
  qualitative_generations.json                       (8.0 KB)

Figures:
  figure1_setup.pdf                                  (24.0 KB)
  figure1_setup.png                                  (164.8 KB)
  figure2_blimp.pdf                                  (19.2 KB)
  figure2_blimp.png                                  (352.3 KB)
  figu

In [5]:
# Key numbers for paper
print("\n" + "=" * 60)
print("KEY NUMBERS FOR PAPER")
print("=" * 60)

# Leaked knowledge count
with open(config.DATA_DIR / 'temporal_facts.json') as f:
    final_data = json.load(f)
print(f"\nDataset sizes:")
for k, v in final_data.items():
    print(f"  {k}: {len(v)}")

# Probing results
if (config.RESULTS_DIR / 'experiment3_probing.json').exists():
    with open(config.RESULTS_DIR / 'experiment3_probing.json') as f:
        probe_data = json.load(f)
    print(f"\nProbing peak accuracies:")
    for model_name, probes in probe_data.items():
        print(f"  {model_name}:")
        for probe_name, probe_info in probes.items():
            best_layer = -1
            best_acc = 0
            for layer, result in probe_info['layer_results'].items():
                if result['mean'] > best_acc:
                    best_acc = result['mean']
                    best_layer = layer
            print(f"    {probe_name}: {best_acc:.3f} (layer {best_layer})")

# Extended analysis
if (config.RESULTS_DIR / 'probe_analysis_extended.json').exists():
    with open(config.RESULTS_DIR / 'probe_analysis_extended.json') as f:
        ext = json.load(f)
    if 'mlp_robustness' in ext:
        print(f"\nMLP vs Linear:")
        for k, v in ext['mlp_robustness'].items():
            print(f"  {k}: linear={v['linear_acc']:.3f}, mlp={v['mlp_acc']:.3f}, delta={v['delta']:+.3f}")
    if 'permutation_baseline' in ext:
        print(f"\nPermutation baselines:")
        for k, v in ext['permutation_baseline'].items():
            print(f"  {k}: real={v['real_acc']:.3f}, permuted={v['perm_mean']:.3f}+/-{v['perm_std']:.3f}, p={v['p_value']:.4f}")

print("\n\nDONE! Download results/ and data/ folders to update the paper.")


KEY NUMBERS FOR PAPER

Dataset sizes:
  pre_1930_true: 200
  pre_1930_false: 199
  post_1930_true: 199
  post_1930_false: 199
  leaked_knowledge: 22

Probing peak accuracies:
  Talkie-1930:
    probe_a_veracity: 0.701 (layer 24)
    probe_b_temporal: 0.982 (layer 16)
    probe_c_knowledge_boundary: 0.960 (layer 16)
  Talkie-Web:
    probe_a_veracity: 0.822 (layer 24)
    probe_b_temporal: 0.995 (layer 20)

MLP vs Linear:
  probe_a_veracity_Talkie-1930: linear=0.701, mlp=0.714, delta=+0.013
  probe_a_veracity_Talkie-Web: linear=0.822, mlp=0.754, delta=-0.068
  probe_b_temporal_Talkie-1930: linear=0.982, mlp=0.970, delta=-0.013
  probe_b_temporal_Talkie-Web: linear=0.995, mlp=0.970, delta=-0.025
  probe_c_knowledge_boundary_Talkie-1930: linear=0.960, mlp=0.936, delta=-0.024

Permutation baselines:
  probe_a_veracity_Talkie-1930: real=0.701, permuted=0.501+/-0.024, p=0.0000
  probe_a_veracity_Talkie-Web: real=0.822, permuted=0.499+/-0.024, p=0.0000
  probe_b_temporal_Talkie-1930: real=0.

In [38]:
import torch, importlib, config
importlib.reload(config)
from model_loader import load_model

w = load_model(config.VINTAGE_MODEL_ID)
m = w.model

lm = m.lm_head.data
print(f"lm_head shape: {lm.shape}, dtype: {lm.dtype}")
print(f"lm_head mean: {lm.float().mean():.8f}, std: {lm.float().std():.8f}")
print(f"lm_head all zeros? {(lm == 0).all()}")
print(f"lm_head nonzero: {(lm != 0).sum()} / {lm.numel()}")
print(f"lm_head_gain: {m.lm_head_gain.w_g.data}")

emb = m.embed.weight.data
print(f"embed shape: {emb.shape}, mean: {emb.float().mean():.8f}, std: {emb.float().std():.8f}")
print(f"embed == lm_head.T? {torch.allclose(emb.float(), lm.T.float(), atol=1e-3)}")

tok_ids = w.tokeniser.encode("The inventor of the telephone was")
inp = torch.tensor([tok_ids], device=config.DEVICE)
logits = m(inp)[0]
ll = logits[0, -1, :]
print(f"logits shape: {logits.shape}, NaN: {logits.isnan().any()}, Inf: {logits.isinf().any()}")
print(f"last logits mean: {ll.mean():.4f}, std: {ll.std():.6f}, min: {ll.min():.4f}, max: {ll.max():.4f}")

topk = ll.topk(10)
print("Top 10 tokens:")
for v, i in zip(topk.values, topk.indices):
    try:
        d = w.tokeniser.decode([i.item()])
    except:
        d = "<err>"
    print(f"  {i.item():6d}: logit={v.item():8.4f} -> {repr(d)}")

[model_loader] Using cached repo: /workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base
[model_loader] Checkpoint: final.ckpt (53.1 GB)
[model_loader] Loading checkpoint into RAM ...
[detect] n_layer=40  n_embd=5120  n_head=40  head_dim=128  intermediate=13696  vocab=65536
[model_loader] Loaded 443/443 params
[model_loader] 13.28B params on cuda (bfloat16)
[tokeniser] Loaded 65536 tokens (max rank 65535)
[model_loader] Tokeniser: vocab=65537
lm_head shape: torch.Size([65536, 5120]), dtype: torch.bfloat16
lm_head mean: 0.00001223, std: 0.00885420
lm_head all zeros? False
lm_head nonzero: 335544320 / 335544320
lm_head_gain: tensor([3.8906], device='cuda:0', dtype=torch.bfloat16)
embed shape: torch.Size([65536, 5120]), mean: 0.00003215, std: 0.01224583


RuntimeError: The size of tensor a (5120) must match the size of tensor b (65536) at non-singleton dimension 1

In [39]:
tok_ids = w.tokeniser.encode("The inventor of the telephone was")
inp = torch.tensor([tok_ids], device=config.DEVICE)
logits = m(inp)[0]
ll = logits[0, -1, :]
print(f"logits shape: {logits.shape}")
print(f"NaN: {logits.isnan().any()}, Inf: {logits.isinf().any()}")
print(f"mean: {ll.mean():.4f}, std: {ll.std():.6f}")
print(f"min: {ll.min():.4f}, max: {ll.max():.4f}")

topk = ll.topk(10)
print("\nTop 10 tokens:")
for v, i in zip(topk.values, topk.indices):
    d = w.tokeniser.decode([i.item()])
    print(f"  {i.item():6d}: logit={v.item():8.4f} -> {repr(d)}")

print("\nBottom 5:")
botk = ll.topk(5, largest=False)
for v, i in zip(botk.values, botk.indices):
    d = w.tokeniser.decode([i.item()])
    print(f"  {i.item():6d}: logit={v.item():8.4f} -> {repr(d)}")

print(f"\nWeight tying check:")
print(f"  embed == lm_head? {torch.allclose(m.embed.weight.data.float(), m.lm_head.data.float(), atol=1e-3)}")

logits shape: torch.Size([1, 6, 65536])
NaN: False, Inf: False
mean: -4.0739, std: 2.491502
min: -42.0000, max: 8.0000

Top 10 tokens:
      32: logit=  8.0000 -> ' '
      46: logit=  7.4062 -> '.'
      45: logit=  6.9375 -> '-'
      44: logit=  6.9375 -> ','
     341: logit=  6.6875 -> 'ch'
     297: logit=  6.5625 -> '\n\n\n'
     286: logit=  6.3438 -> ' in'
      42: logit=  6.1250 -> '*'
     105: logit=  6.0000 -> 'i'
     290: logit=  6.0000 -> ' to'

Bottom 5:
       3: logit=-42.0000 -> '\x03'
       2: logit=-42.0000 -> '\x02'
       6: logit=-42.0000 -> '\x06'
      11: logit=-42.0000 -> '\x0b'
       8: logit=-42.0000 -> '\x08'

Weight tying check:
  embed == lm_head? False


In [40]:
import torch.nn.functional as F

tok_ids = w.tokeniser.encode("The inventor of the telephone was")
inp = torch.tensor([tok_ids], device=config.DEVICE)

# Get final hidden state manually
x = m.embed(inp)
x = F.rms_norm(x, (x.shape[-1],))
e_x = x
for block in m.blocks:
    x = block(e_x, x, (m.cos[:,:inp.shape[1]].to(inp.device), m.sin[:,:inp.shape[1]].to(inp.device)))
x = F.rms_norm(x, (x.shape[-1],))
h = x[0, -1, :]

# Test 3 options
print("=== Option A: current lm_head (broken) ===")
la = F.linear(h.unsqueeze(0), m.lm_head_gain(m.lm_head)).float().squeeze()
for v, i in zip(*la.topk(5)):
    print(f"  {i.item():6d}: {v.item():7.3f} -> {repr(w.tokeniser.decode([i.item()]))}")

print("\n=== Option B: embed.weight (weight tying) ===")
lb = F.linear(h.unsqueeze(0), m.embed.weight).float().squeeze()
for v, i in zip(*lb.topk(5)):
    print(f"  {i.item():6d}: {v.item():7.3f} -> {repr(w.tokeniser.decode([i.item()]))}")

print("\n=== Option C: embed.weight * gain ===")
lc = F.linear(h.unsqueeze(0), m.lm_head_gain(m.embed.weight)).float().squeeze()
for v, i in zip(*lc.topk(5)):
    print(f"  {i.item():6d}: {v.item():7.3f} -> {repr(w.tokeniser.decode([i.item()]))}")

=== Option A: current lm_head (broken) ===
      32:   8.000 -> ' '
      46:   7.406 -> '.'
      45:   6.938 -> '-'
      44:   6.938 -> ','
     341:   6.688 -> 'ch'

=== Option B: embed.weight (weight tying) ===
   23212:   3.766 -> 'Chief'
   42646:   3.234 -> ' Fart'
   57503:   3.188 -> 'Prior'
   61965:   3.094 -> 'SOLD'
   21659:   3.094 -> ' BROAD'

=== Option C: embed.weight * gain ===
   23212:  14.625 -> 'Chief'
   42646:  12.562 -> ' Fart'
   57503:  12.438 -> 'Prior'
   61965:  12.062 -> 'SOLD'
   21659:  12.000 -> ' BROAD'


In [41]:
# 1. Does the tokenizer work correctly?
toks = w.tokeniser.encode("The inventor of the telephone was")
print("Tokens:", toks)
for t in toks:
    print(f"  {t}: {repr(w.tokeniser.decode([t]))}")

# 2. Does log-likelihood scoring work?
ll1 = w.log_likelihood("The cat sat on the mat.")
ll2 = w.log_likelihood("mat the on cat sat The.")
print(f"\nLL grammatical: {ll1:.2f}")
print(f"LL scrambled:   {ll2:.2f}")
print(f"Correct: {ll1 > ll2}")

# 3. Hidden state norms per layer
inp = torch.tensor([toks], device=config.DEVICE)
_, hs = m(inp, output_hidden_states=True)
print(f"\nHidden state norms per layer:")
for i, h in enumerate(hs):
    n = h[0, -1].float().norm().item()
    std = h[0, -1].float().std().item()
    print(f"  Layer {i:2d}: norm={n:.2f}  std={std:.6f}")

# 4. Check embed_skip gains (are they huge?)
print(f"\nEmbed skip gains:")
for i, block in enumerate(m.blocks):
    g = block.embed_skip.a_g.item()
    if abs(g) > 0.01:
        print(f"  Block {i}: embed_skip={g:.4f}")

Tokens: [486, 21789, 270, 260, 9462, 359]
  486: 'The'
  21789: ' inventor'
  270: ' of'
  260: ' the'
  9462: ' telephone'
  359: ' was'

LL grammatical: -49.24
LL scrambled:   -49.17
Correct: False

Hidden state norms per layer:
  Layer  0: norm=9.53  std=0.133138
  Layer  1: norm=5.78  std=0.080770
  Layer  2: norm=2.49  std=0.034784
  Layer  3: norm=0.16  std=0.002191
  Layer  4: norm=0.21  std=0.002874
  Layer  5: norm=0.26  std=0.003625
  Layer  6: norm=0.29  std=0.004010
  Layer  7: norm=0.33  std=0.004630
  Layer  8: norm=0.36  std=0.005024
  Layer  9: norm=0.41  std=0.005660
  Layer 10: norm=0.51  std=0.007187
  Layer 11: norm=1.20  std=0.016738
  Layer 12: norm=1.49  std=0.020769
  Layer 13: norm=1.80  std=0.025189
  Layer 14: norm=2.06  std=0.028822
  Layer 15: norm=2.40  std=0.033557
  Layer 16: norm=3.02  std=0.042167
  Layer 17: norm=4.47  std=0.062419
  Layer 18: norm=7.29  std=0.101943
  Layer 19: norm=10.62  std=0.148471
  Layer 20: norm=14.38  std=0.201039
  Layer 21:

In [42]:
import os
repo = "/workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base"
print("Files in repo:")
for f in sorted(os.listdir(repo)):
    size = os.path.getsize(os.path.join(repo, f))
    print(f"  {f:40s} {size/1024:.1f} KB")

# Check for config or model code
for name in ["config.json", "README.md", "model.py", "config.yaml"]:
    path = os.path.join(repo, name)
    if os.path.exists(path):
        print(f"\n=== {name} ===")
        with open(path) as f:
            print(f.read()[:3000])

Files in repo:
  .cache                                   60.4 KB
  .gitattributes                           1.5 KB
  README.md                                0.4 KB
  final.ckpt                               51876242.7 KB
  vocab.txt                                4484.2 KB

=== README.md ===
---
language:
- en
license: apache-2.0
---

# talkie-1930-13b-base

talkie-1930-13b-base is a 13B vintage language model trained on 260B tokens of pre-1931 English-language text. An instruction-tuned version of the model is available at talkie-lm/talkie-1930-13b-it.

Read more about talkie in our [report](https://talkie-lm.com/). 

Reference code to run talkie is available on [GitHub](https://github.com/talkie-lm/talkie).


In [43]:
# Reference order:
if "model_state_dict" in ckpt: ...
elif "model" in ckpt: ...

# Our order:
for wrapper in ["model", "state_dict", "model_state_dict"]:  # ← wrong priority!

SyntaxError: incomplete input (1916690767.py, line 6)

In [44]:
!pip install git+https://github.com/talkie-lm/talkie.git -q


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import tiktoken

tokeniser = tiktoken.get_encoding("gpt2")
prompt = "The inventor of the telephone was"
tok_ids = tokeniser.encode(prompt)
inp = torch.tensor([tok_ids], device="cuda")

with torch.no_grad():
    logits = ref_model(inp)

print(f"Logits shape: {logits.shape}")
topk = logits[0].topk(10)
print("Top 10 tokens:")
for v, i in zip(topk.values, topk.indices):
    d = tokeniser.decode([i.item()])
    print(f"  {i.item():6d}: {v.item():7.3f} -> {repr(d)}")

Logits shape: torch.Size([1, 65536])
Top 10 tokens:
      32:   8.375 -> 'A'
      46:   7.281 -> 'O'
      45:   6.969 -> 'N'
      44:   6.781 -> 'M'
     297:   6.438 -> 'll'
     286:   6.406 -> ' of'
     302:   6.375 -> ' re'
     640:   6.344 -> ' time'
     523:   6.312 -> ' so'
     340:   6.281 -> ' it'


In [3]:
# Check if talkie package provides its own tokenizer
import talkie
print(dir(talkie))

# Check if there's a tokenizer in the model or package
from talkie import model as talkie_model
print([x for x in dir(talkie_model) if 'tok' in x.lower() or 'vocab' in x.lower() or 'encode' in x.lower()])

# Also check the repo structure
import importlib
spec = importlib.util.find_spec("talkie")
print(f"Talkie package location: {spec.origin}")

['GenerationConfig', 'GenerationResult', 'MODELS', 'Message', 'ModelSpec', 'Talkie', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_warnings', 'chat', 'config', 'download', 'download_model', 'format_chat', 'format_prompt', 'generate', 'get_model_files', 'model', 'sampling', 'tokenizer']
[]
Talkie package location: /usr/local/lib/python3.11/dist-packages/talkie/__init__.py


In [4]:
from talkie import tokenizer
print(dir(tokenizer))

# Try to get the actual tokenizer
tok = tokenizer.get_tokenizer() if hasattr(tokenizer, 'get_tokenizer') else None
if tok is None:
    # Check what's available
    print([x for x in dir(tokenizer) if not x.startswith('_')])

['BASE_VOCAB_SIZE', 'IT_VOCAB_SIZE', 'Path', '_BASE_SPECIAL_TOKENS', '_IT_SPECIAL_TOKENS', '_PAT_STR', '__annotations__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'annotations', 'build_tokenizer', 'load_tiktoken_bpe', 'tiktoken']
['BASE_VOCAB_SIZE', 'IT_VOCAB_SIZE', 'Path', 'annotations', 'build_tokenizer', 'load_tiktoken_bpe', 'tiktoken']


In [5]:
# Also check the checkpoint directory listing
import os
cache_dir = "/workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base/"
print(os.listdir(cache_dir))

['final.ckpt', '.gitattributes', 'vocab.txt', 'README.md', '.cache']


In [6]:
from talkie.tokenizer import build_tokenizer, BASE_VOCAB_SIZE

print(f"BASE_VOCAB_SIZE: {BASE_VOCAB_SIZE}")

# Build tokenizer with the vocab file
vocab_path = "/workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base/vocab.txt"
tok = build_tokenizer(vocab_path)
print(f"Tokenizer type: {type(tok)}")
print(f"Vocab size: {tok.n_vocab if hasattr(tok, 'n_vocab') else 'unknown'}")

# Now test with correct tokenizer
prompt = "The inventor of the telephone was"
tok_ids = tok.encode(prompt)
print(f"Token IDs: {tok_ids}")
inp = torch.tensor([tok_ids], device="cuda")

with torch.no_grad():
    logits = ref_model(inp)

print(f"\nLogits shape: {logits.shape}")
topk = logits[0].topk(10)
print("Top 10 tokens:")
for v, i in zip(topk.values, topk.indices):
    d = tok.decode([i.item()])
    print(f"  {i.item():6d}: {v.item():7.3f} -> {repr(d)}")

BASE_VOCAB_SIZE: 65536
Tokenizer type: <class 'tiktoken.core.Encoding'>
Vocab size: 65536
Token IDs: [486, 21789, 270, 260, 9462, 359]

Logits shape: torch.Size([1, 65536])
Top 10 tokens:
      32:   8.000 -> ' '
      46:   7.406 -> '.'
      45:   6.938 -> '-'
      44:   6.938 -> ','
     341:   6.688 -> 'ch'
     297:   6.562 -> '\n\n\n'
     286:   6.344 -> ' in'
      42:   6.125 -> '*'
     105:   6.000 -> 'i'
     290:   6.000 -> ' to'


In [7]:
import os

# Check checkpoint size (13B model in bf16 should be ~26GB)
ckpt_path = "/workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base/final.ckpt"
size_gb = os.path.getsize(ckpt_path) / (1024**3)
print(f"Checkpoint size: {size_gb:.2f} GB")

# Check hidden state norms with reference model
x = inp
with torch.no_grad():
    e_x = ref_model.embed(x)
    e_x = torch.nn.functional.rms_norm(e_x, (e_x.shape[-1],))
    cos_sin = ref_model.rope(e_x)
    h = e_x.clone()
    for i, block in enumerate(ref_model.blocks):
        h = block(e_x, h, cos_sin)
        if i < 5 or i >= 38:
            print(f"Layer {i:2d}: norm = {h.norm(dim=-1).mean().item():.4f}")

Checkpoint size: 49.47 GB


AttributeError: 'TalkieModel' object has no attribute 'rope'

In [8]:
# Check model structure
print([name for name, _ in ref_model.named_children()])

['embed', 'blocks', 'lm_head_gain']


In [9]:
# Try the package's generate function
from talkie import generate
result = generate(ref_model, tok, "The inventor of the telephone was", max_new_tokens=20)
print(f"Generated: {result}")

TypeError: 'module' object is not callable

In [10]:
from talkie import generate as gen_module
print([x for x in dir(gen_module) if not x.startswith('_')])

['GenerationConfig', 'GenerationResult', 'Generator', 'IT_VOCAB_SIZE', 'Literal', 'MODELS', 'Message', 'STOP_WINDOW', 'Talkie', 'annotations', 'build_tokenizer', 'dataclass', 'format_chat', 'format_prompt', 'get_model_files', 'list_top_k_tensor', 'list_top_p_tensor', 'load_checkpoint', 'scalar_top_k_tensor', 'scalar_top_p_tensor', 'torch', 'truncate_at_stop']


In [11]:
# Check hidden state norms with correct attribute names
with torch.no_grad():
    e_x = ref_model.embed(inp)
    e_x = torch.nn.functional.rms_norm(e_x, (e_x.shape[-1],))
    
    # Find rope - might be inside blocks
    block0 = ref_model.blocks[0]
    print(f"Block attributes: {[n for n,_ in block0.named_children()]}")
    print(f"Block all attrs: {[a for a in dir(block0) if not a.startswith('_')]}")

Block attributes: ['attn', 'attn_gain', 'mlp', 'mlp_gain', 'embed_skip']
Block all attrs: ['T_destination', 'add_module', 'apply', 'attn', 'attn_gain', 'bfloat16', 'buffers', 'call_super_init', 'children', 'compile', 'cpu', 'cuda', 'double', 'dump_patches', 'embed_skip', 'eval', 'extra_repr', 'float', 'forward', 'get_buffer', 'get_extra_state', 'get_parameter', 'get_submodule', 'half', 'ipu', 'load_state_dict', 'mlp', 'mlp_gain', 'modules', 'named_buffers', 'named_children', 'named_modules', 'named_parameters', 'parameters', 'register_backward_hook', 'register_buffer', 'register_forward_hook', 'register_forward_pre_hook', 'register_full_backward_hook', 'register_full_backward_pre_hook', 'register_load_state_dict_post_hook', 'register_module', 'register_parameter', 'register_state_dict_pre_hook', 'requires_grad_', 'set_extra_state', 'share_memory', 'state_dict', 'to', 'to_empty', 'train', 'training', 'type', 'xpu', 'zero_grad']


In [12]:
# Try Generator class
from talkie.generate import Generator
gen = Generator(ref_model, tok)
result = gen.generate("The inventor of the telephone was", max_new_tokens=20)
print(f"Type: {type(result)}")
print(f"Result: {result}")

TypeError: Generator() takes no arguments

In [13]:
# Trace the full forward pass to check hidden state norms
import torch.nn.functional as F

with torch.no_grad():
    # Step through the model's own forward method
    e_x = ref_model.embed(inp)
    e_x = F.rms_norm(e_x, (e_x.shape[-1],))
    
    # RoPE must be computed somewhere - check the model's forward source
    import inspect
    print(inspect.getsource(ref_model.forward))

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Run a forward pass and return ``[B, V]`` logits for the last position."""
        _, seq_len = input_ids.shape
        cos_sin = self.cos[:, :seq_len], self.sin[:, :seq_len]

        x = self.embed(input_ids)
        x = F.rms_norm(x, (x.shape[-1],))
        e_x = x
        for block in self.blocks:
            x = block(e_x, x, cos_sin)
        x = F.rms_norm(x, (x.shape[-1],))

        return F.linear(x[:, -1, :], self.lm_head_gain(self.lm_head)).float()



In [14]:
import inspect
print(inspect.getsource(ref_model.forward))

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Run a forward pass and return ``[B, V]`` logits for the last position."""
        _, seq_len = input_ids.shape
        cos_sin = self.cos[:, :seq_len], self.sin[:, :seq_len]

        x = self.embed(input_ids)
        x = F.rms_norm(x, (x.shape[-1],))
        e_x = x
        for block in self.blocks:
            x = block(e_x, x, cos_sin)
        x = F.rms_norm(x, (x.shape[-1],))

        return F.linear(x[:, -1, :], self.lm_head_gain(self.lm_head)).float()



In [15]:
import torch.nn.functional as F

with torch.no_grad():
    _, seq_len = inp.shape
    cos_sin = ref_model.cos[:, :seq_len], ref_model.sin[:, :seq_len]
    
    x = ref_model.embed(inp)
    x = F.rms_norm(x, (x.shape[-1],))
    e_x = x.clone()
    
    for i, block in enumerate(ref_model.blocks):
        x = block(e_x, x, cos_sin)
        if i < 5 or i >= 38:
            print(f"Layer {i:2d}: norm = {x.norm(dim=-1).mean().item():.4f}")
    
    x_final = F.rms_norm(x, (x.shape[-1],))
    logits = F.linear(x_final[:, -1, :], ref_model.lm_head_gain(ref_model.lm_head)).float()
    
    print(f"\nPre-norm last hidden norm: {x[:, -1, :].norm().item():.4f}")
    print(f"Post-norm last hidden norm: {x_final[:, -1, :].norm().item():.4f}")
    print(f"lm_head_gain: {[p.data for p in ref_model.lm_head_gain.parameters()]}")
    print(f"Logits std: {logits.std().item():.4f}")

Layer  0: norm = 5.6875
Layer  1: norm = 2.4375
Layer  2: norm = 0.1670
Layer  3: norm = 0.2393
Layer  4: norm = 0.2891
Layer 38: norm = 680.0000
Layer 39: norm = 1080.0000

Pre-norm last hidden norm: 1072.0000
Post-norm last hidden norm: 72.0000
lm_head_gain: [tensor([3.8906], device='cuda:0', dtype=torch.bfloat16)]
Logits std: 2.4915


In [1]:
import torch
from talkie.model import load_checkpoint

ckpt_path = "/workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base/final.ckpt"
ref_model = load_checkpoint(ckpt_path, torch.device("cuda"))

# Test generation with reference model
prompt = "The inventor of the telephone was"
tok_ids = w.tokeniser.encode(prompt)
inp = torch.tensor([tok_ids], device="cuda")
logits = ref_model(inp)
print(f"Logits shape: {logits.shape}")
topk = logits[0].topk(10)
print("Top 10 tokens:")
for v, i in zip(topk.values, topk.indices):
    d = w.tokeniser.decode([i.item()])
    print(f"  {i.item():6d}: {v.item():7.3f} -> {repr(d)}")

/usr/local/lib/python3.11/dist-packages/talkie/model.py:275: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, map_location="cpu")


NameError: name 'w' is not defined

In [46]:
import torch
from talkie.model import load_checkpoint

ckpt_path = "/workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base/final.ckpt"
ref_model = load_checkpoint(ckpt_path, torch.device("cuda"))

prompt = "The inventor of the telephone was"
tok_ids = w.tokeniser.encode(prompt)
inp = torch.tensor([tok_ids], device="cuda")

with torch.no_grad():
    logits = ref_model(inp)

print(f"Logits shape: {logits.shape}")
topk = logits[0].topk(10)
print("Top 10 tokens:")
for v, i in zip(topk.values, topk.indices):
    d = w.tokeniser.decode([i.item()])
    print(f"  {i.item():6d}: {v.item():7.3f} -> {repr(d)}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 640.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 88.94 MiB is free. Including non-PyTorch memory, this process has 79.15 GiB memory in use. Of the allocated memory 78.36 GiB is allocated by PyTorch, and 307.91 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [17]:
# Greedy generation with reference model + correct tokenizer
import torch.nn.functional as F

prompt = "The inventor of the telephone was"
ids = tok.encode(prompt)

with torch.no_grad():
    for step in range(100):
        inp_t = torch.tensor([ids], device="cuda")
        logits = ref_model(inp_t)
        next_id = logits[0].argmax().item()
        ids.append(next_id)

generated = tok.decode(ids)
print(f"Generated text:\n{generated}")

Generated text:
The inventor of the telephone was                                                                                                    


In [18]:
# Also try a prompt that looks more like pre-1931 book text
prompt2 = "It was a dark and stormy night, and the"
ids2 = tok.encode(prompt2)

with torch.no_grad():
    for step in range(100):
        inp_t = torch.tensor([ids2], device="cuda")
        logits = ref_model(inp_t)
        next_id = logits[0].argmax().item()
        ids2.append(next_id)

generated2 = tok.decode(ids2)
print(f"Generated text 2:\n{generated2}")

Generated text 2:
It was a dark and stormy night, and the                                                                                                    


In [19]:
import torch
ckpt_path = "/workspace/Talkie/model_cache/talkie-lm--talkie-1930-13b-base/final.ckpt"
raw = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print(f"Type: {type(raw)}")
if isinstance(raw, dict):
    print(f"Top-level keys: {list(raw.keys())}")
    for k, v in raw.items():
        if isinstance(v, dict):
            print(f"  '{k}': {len(v)} sub-keys, first 5: {list(v.keys())[:5]}")
        elif isinstance(v, torch.Tensor):
            print(f"  '{k}': shape={v.shape}, dtype={v.dtype}")
        else:
            print(f"  '{k}': {type(v).__name__} = {v}")
            

Type: <class 'dict'>
Top-level keys: ['model']
  'model': 443 sub-keys, first 5: ['_orig_mod.lm_head', '_orig_mod.embed.weight', '_orig_mod.blocks.0.attn.attn_query.weight', '_orig_mod.blocks.0.attn.attn_key.weight', '_orig_mod.blocks.0.attn.attn_value.weight']


In [20]:
# Check checkpoint weight dtype
state = raw["model"]
sample_key = "_orig_mod.lm_head"
print(f"Checkpoint lm_head dtype: {state[sample_key].dtype}, shape: {state[sample_key].shape}")
print(f"Checkpoint embed dtype: {state['_orig_mod.embed.weight'].dtype}")

# Check what the loaded model has
print(f"\nLoaded model lm_head dtype: {ref_model.lm_head.dtype}")
print(f"Loaded model embed dtype: {ref_model.embed.weight.dtype}")

Checkpoint lm_head dtype: torch.float32, shape: torch.Size([65536, 5120])
Checkpoint embed dtype: torch.float32

Loaded model lm_head dtype: torch.bfloat16
Loaded model embed dtype: torch.bfloat16


In [21]:
# Try fp32 inference instead of bf16
ref_model_fp32 = ref_model.float()

prompt = "The inventor of the telephone was"
ids = tok.encode(prompt)
inp_t = torch.tensor([ids], device="cuda")

with torch.no_grad():
    logits_fp32 = ref_model_fp32(inp_t)

topk = logits_fp32[0].topk(10)
print("FP32 Top 10:")
for v, i in zip(topk.values, topk.indices):
    print(f"  {i.item():6d}: {v.item():7.3f} -> {repr(tok.decode([i.item()]))}")

FP32 Top 10:
     258:  10.851 -> ' a'
    3021:  10.256 -> ' born'
     396:   9.991 -> ' not'
     260:   9.722 -> ' the'
     286:   9.479 -> ' in'
     357:   9.229 -> ' an'
    9958:   8.783 -> ' talking'
     494:   8.511 -> ' one'
     949:   8.480 -> ' present'
     361:   8.471 -> ' at'


In [22]:
# Greedy generation in fp32
prompt = "The inventor of the telephone was"
ids = list(tok.encode(prompt))

with torch.no_grad():
    for step in range(50):
        inp_t = torch.tensor([ids], device="cuda")
        logits = ref_model_fp32(inp_t)
        next_id = logits[0].argmax().item()
        ids.append(next_id)

print(tok.decode(ids))

The inventor of the telephone was a poor man, and he had to work hard to get the money to pay for the patent. He had to sell his patent to a company, and he got only $50,000 for it. He was a very poor man, and he


In [23]:
# Quick check: what dtype does our model_loader use?
import subprocess
result = subprocess.run(['grep', '-n', 'bfloat\|float16\|half\|\.to(', '/workspace/Talkie/model_loader.py'], capture_output=True, text=True)
print(result.stdout)

179:        cos_sin = self.cos[:, :T].to(input_ids.device), self.sin[:, :T].to(input_ids.device)
574:    model = model.to(dtype=torch.bfloat16).to(config.DEVICE)
577:    print(f"[model_loader] {n_params/1e9:.2f}B params on {config.DEVICE} (bfloat16)")



In [24]:
with open("/workspace/Talkie/model_loader.py", "r") as f:
    for i, line in enumerate(f.readlines(), 1):
        if any(kw in line.lower() for kw in ['bfloat', 'float16', 'half', 'dtype', '.to(']):
            print(f"{i}: {line.rstrip()}")

77:     inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
78:     t = torch.arange(seq_len, dtype=torch.float32)
179:         cos_sin = self.cos[:, :T].to(input_ids.device), self.sin[:, :T].to(input_ids.device)
253:     print(f"\n{'Key':<70} {'Shape':<25} Dtype")
257:         d = v.dtype if hasattr(v, "dtype") else type(v).__name__
337:         self.dtype = config.DTYPE
386:             padded = torch.zeros(len(batch_ids), max_len, dtype=torch.long,
389:                 padded[j, :len(ids)] = torch.tensor(ids, dtype=torch.long)
423:             padded = torch.zeros(B, max_len, dtype=torch.long, device=self.device)
425:                 padded[j, :len(ids)] = torch.tensor(ids, dtype=torch.long)
510:             padded = torch.zeros(B, max_len, dtype=torch.long, device=self.device)
512:                 padded[j, :len(ids)] = torch.tensor(ids, dtype=torch.long)
574:     model = model.to(dtype=torch.bfloat16).to(config.DEVICE)
577:     print(f"[model_